# Prospective Donor Persona Discovery
## Refactored collaborative experiment notebook

This notebook separates the **stable engine**, **fixed evidence**, and **experiment surface**. The default working candidate is the prospect-only, no-education, down-weighted `k=3` architecture, but no persona names or production assignments are frozen here.


# Zone 0 — Engine
**Stable; rarely edited.** These cells contain imports, data contracts, registries, transforms, distance math, fitting, profiling, and integrity utilities. Collaborators should use the PARAMS panels in Zone 2 instead.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

NOTEBOOK_VERSION = '3.0-refactor'
SEGMENT_ENGINE_VERSION = '2.0-refactor'
RUN_DATE_UTC = datetime.now(timezone.utc).isoformat()
SEED = 17
KMEDOIDS_N_INIT = 8
KMEDOIDS_MAX_ITER = 50
STABILITY_FRACTION = 0.80

RAW_FILENAME = 'Donor Segmentation Survey - Clean Raw Data (PII Removed) - DonorsChoose - September 2024.xlsx - Export.csv'
PCA_FILENAME = 'DonorsChoose Cluster Analysis Details.xlsx - PCA Factor Loadings.csv'
SURVEY_FILENAME = 'FINAL_Donor Segmentation Survey - Final for Fielding.docx.pdf'
HANDOFF_FILENAME = 'Donor_Persona_Analysis_Engineering_Handoff.docx'

candidate_dirs = [
    Path(os.getenv('DONOR_PERSONA_DATA_DIR', '.')).expanduser().resolve(),
    Path('/mnt/data'),
]
DATA_DIR = next((path for path in candidate_dirs if (path / RAW_FILENAME).exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        f'Could not find {RAW_FILENAME!r}. Set DONOR_PERSONA_DATA_DIR to the source folder.'
    )

OUTPUT_DIR = Path(
    os.getenv('DONOR_PERSONA_OUTPUT_DIR', DATA_DIR / 'persona_outputs_refactored')
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = DATA_DIR / RAW_FILENAME
PCA_PATH = DATA_DIR / PCA_FILENAME
SURVEY_PATH = DATA_DIR / SURVEY_FILENAME
HANDOFF_PATH = DATA_DIR / HANDOFF_FILENAME

EXPERIMENT_MODES = {
    # The full-sample fit is fixed across modes so changing resample depth never changes the candidate itself.
    'smoke': {'stability_runs': 3, 'fit_n_init': KMEDOIDS_N_INIT, 'stability_n_init': 2},
    '20': {'stability_runs': 20, 'fit_n_init': KMEDOIDS_N_INIT, 'stability_n_init': 4},
    '50': {'stability_runs': 50, 'fit_n_init': KMEDOIDS_N_INIT, 'stability_n_init': 4},
    '100': {'stability_runs': 100, 'fit_n_init': KMEDOIDS_N_INIT, 'stability_n_init': 5},
}


In [2]:
Q_IMPORTANCE = 'How would you rate the importance of each of the following attributes when selecting an organization/cause to make a charitable donation to?'
Q_MOTIVATIONS = 'Which of the following motivates you to make a charitable donation? Select all that apply.'
Q_IMPACT_SCALE = 'Which of the following best describes the scale at which you prefer your charitable donations to make an impact?'
Q_CURRENT_CAUSES = 'Which of the following causes/organizations do you typically make charitable donations to? Select all that apply.'
Q_OPENNESS = 'You indicated that you do not currently make charitable donations to the following causes/organizations. How open are you to making a charitable donation to these causes in the future?'
Q_EDU_FORMATS = 'Which of the following types of education-related charitable donations would you be most likely to make? Select up to two (2).'
Q_DISCOVERY = 'Which, if any, of these sources have you used to discover new organizations or causes to donate to? Select all that apply.'
Q_RESEARCH_SOURCES = 'What sources of information do you use when researching organizations/causes to make a charitable donation? Select all that apply.'
Q_RESEARCH_INFO = 'What kinds of information did you look for when researching organizations/causes to make a charitable donation? Select all that apply.'
Q_CHALLENGES = 'Which of the following challenges do you encounter when making a charitable donation? Select all that apply.'
Q_MESSAGING = 'How influential is the following messaging when deciding to make a charitable donation?'
Q_AD_RESPONSE = 'Have you ever made a charitable donation in response to any of the following forms of advertising/messaging? Select your top three (3).'
Q_CHARITY_ATTITUDES = 'To what extent do you agree with each of the following statements about charitable donations?'

FREQUENCY_COL = 'How often do you make charitable donations?'
AMOUNT_COL = 'What is the average donation amount you typically donate to charitable organizations/causes? Your best guess is fine.'
ORG_COUNT_COL = 'How many charitable organizations, on average, do you donate to in a given year?'
DONATION_COUNT_COL = 'On average, how many times per year do you make an individual charitable donation?'
DECISION_COL = 'Which of the following best describes you at the point before you make a charitable donation to an organization?'

IMPORTANT_MAP = {
    'Not at all important': 1,
    'Slightly important': 2,
    'Moderately important': 3,
    'Very important': 4,
    'Extremely important': 5,
}
INFLUENTIAL_MAP = {
    'Not at all influential': 1,
    'Slightly influential': 2,
    'Moderately influential': 3,
    'Very influential': 4,
    'Extremely influential': 5,
}
AGREE_MAP = {
    'Strongly disagree': 1,
    'Somewhat disagree': 2,
    'Neither agree nor disagree': 3,
    'Somewhat agree': 4,
    'Strongly agree': 5,
}
OPEN_MAP = {
    'Not at all open': 1,
    'Slightly open': 2,
    'Moderately open': 3,
    'Very open': 4,
    'Extremely open': 5,
}
FREQUENCY_MAP = {
    'Once a year': 1,
    'Once every 4 to 6 months': 2,
    'Once every 2 to 3 months': 3,
    'Once a month': 4,
    'More often than once a month': 5,
}
AMOUNT_MAP = {
    'Less than $10': 1,
    '$10-$19': 2,
    '$20-$49': 3,
    '$50-$99': 4,
    '$100 - $499': 5,
    '$500 - $999': 6,
    '$1,000 - $9,999': 7,
    '$10,000 or more': 8,
}

DERIVED_SUFFIXES = {
    *IMPORTANT_MAP,
    *INFLUENTIAL_MAP,
    *AGREE_MAP,
    *OPEN_MAP,
    'Top 2',
}


def likert_col(item: str, question: str) -> str:
    col = f'{item} - {question}'
    if col not in raw.columns:
        raise KeyError(col)
    return col


def option_col(question: str, option: str) -> str:
    col = f'{question} - {option}'
    if col not in raw.columns:
        raise KeyError(col)
    return col


def selected(frame: pd.DataFrame, question: str, option: str) -> pd.Series:
    return frame[option_col(question, option)].notna()


# DESCRIPTOR / PROFILE-ONLY — never clustering inputs unless explicitly registered as a DistanceUnit.
AGE_COL = 'How old are you?'
EMPLOYMENT_COL = 'Which of the following best describes your current employment status?'
EDUCATION_COL = 'What is the highest level of education you have completed?'
INCOME_COL = 'What is your average annual household income?'
GENDER_COL = 'What is your gender identity?'
REGION_COL = 'Region Regrouped: In which state do you currently reside?'
URBANICITY_COL = 'Which of the following best describes where you live?'
RACE_QUESTION = 'Which of the following best describes your race? Select all that apply.'
CHILDREN_QUESTION = 'Do children in any of the following age groups currently live in your home? Please select all that apply.'
Q_MEDIA = 'Which of the following sources of media do you read, watch, or listen to on a regular basis? Select all that apply.'
Q_SOCIAL_PERSONAL = 'Which of the following social media sites do you use for personal purposes? Select all that apply.'
Q_SOCIAL_NEWS = 'Which of the following social media sites do you use for sources of news? Select all that apply.'
Q_SCHOOL_INVOLVEMENT = "Which of the following activities are/were you involved in at your child(ren)'s school? Select all that apply."

BLOCK_NAMES = (
    'giving_commitment',
    'giving_job_impact',
    'trust_research_friction',
    'trigger_channel',
    'education_receptivity',
)


In [3]:
def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def frame_hash(frame: pd.DataFrame) -> str:
    payload = pd.util.hash_pandas_object(frame, index=True).to_numpy().tobytes()
    return hashlib.sha256(payload).hexdigest()


def array_hash(array: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(array).view(np.uint8)).hexdigest()


def object_snapshot(**objects: Any) -> dict[str, str]:
    snapshot = {}
    for name, value in objects.items():
        if isinstance(value, pd.DataFrame):
            snapshot[name] = frame_hash(value)
        elif isinstance(value, pd.Series):
            snapshot[name] = frame_hash(value.to_frame())
        elif isinstance(value, np.ndarray):
            snapshot[name] = array_hash(value)
        else:
            snapshot[name] = hashlib.sha256(repr(value).encode()).hexdigest()
    return snapshot


def integrity_gate(snapshot: Mapping[str, str], **objects: Any) -> None:
    observed = object_snapshot(**objects)
    assert observed == dict(snapshot), {
        key: (snapshot.get(key), observed.get(key))
        for key in set(snapshot) | set(observed)
        if snapshot.get(key) != observed.get(key)
    }


def write_manifest(name: str, params: Mapping[str, Any], outputs: Sequence[Path]) -> Path:
    path = OUTPUT_DIR / f'{name}_manifest.json'
    payload = {
        'notebook_version': NOTEBOOK_VERSION,
        'engine_version': SEGMENT_ENGINE_VERSION,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'params': dict(params),
        'input_sha256': {row['artifact']: row['sha256'] for _, row in input_manifest.iterrows()},
        'outputs': [str(item.relative_to(OUTPUT_DIR)) for item in outputs if item.exists()],
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    return path


def print_decision_table(frame: pd.DataFrame, columns: Sequence[str] | None = None) -> None:
    view = frame if columns is None else frame.loc[:, list(columns)]
    print(view.to_string(index=False))


raw = pd.read_csv(RAW_PATH, low_memory=False)
pca_loadings_raw = pd.read_csv(PCA_PATH)

QC_COL = 'For quality control purposes, please select "Somewhat disagree" - To what extent do you agree with each of the following statements about charitable donations?'
ID_COL = 'responseidkey'
SOURCE_COL = 'Source'
HISTORY_COL = 'DonorsChoose Donation History'
LEGACY_CLUSTER_COL = 'Four Cluster Solution'

required_columns = {ID_COL, SOURCE_COL, HISTORY_COL, LEGACY_CLUSTER_COL, QC_COL}
missing_required = required_columns.difference(raw.columns)
assert not missing_required, f'Missing required columns: {sorted(missing_required)}'
assert raw.shape == (1438, 884), f'Unexpected raw shape: {raw.shape}'
assert raw[ID_COL].is_unique, 'Respondent key is not unique.'

attention_pass = raw[QC_COL].eq('Somewhat disagree')
retained = raw.loc[attention_pass].copy()
prospect_mask_retained = retained[SOURCE_COL].eq('Panel')
donor_mask_retained = retained[SOURCE_COL].eq('List')

assert (len(retained), int(prospect_mask_retained.sum()), int(donor_mask_retained.sum())) == (1245, 1120, 125)
assert raw[HISTORY_COL].eq('Past donor (List)').equals(raw[SOURCE_COL].eq('List'))
assert retained[HISTORY_COL].eq('Past donor (List)').equals(retained[SOURCE_COL].eq('List'))

input_manifest = pd.DataFrame([
    {'artifact': 'raw survey export', 'path': str(RAW_PATH), 'sha256': sha256_file(RAW_PATH), 'bytes': RAW_PATH.stat().st_size},
    {'artifact': 'PCA factor loadings', 'path': str(PCA_PATH), 'sha256': sha256_file(PCA_PATH), 'bytes': PCA_PATH.stat().st_size},
    {'artifact': 'survey instrument', 'path': str(SURVEY_PATH), 'sha256': sha256_file(SURVEY_PATH) if SURVEY_PATH.exists() else None, 'bytes': SURVEY_PATH.stat().st_size if SURVEY_PATH.exists() else None},
    {'artifact': 'analysis handoff', 'path': str(HANDOFF_PATH), 'sha256': sha256_file(HANDOFF_PATH) if HANDOFF_PATH.exists() else None, 'bytes': HANDOFF_PATH.stat().st_size if HANDOFF_PATH.exists() else None},
])
_input_hashes = dict(zip(input_manifest['artifact'], input_manifest['sha256']))
print(
    f'Notebook {NOTEBOOK_VERSION} | run {RUN_DATE_UTC} | seed {SEED} | '
    f'raw={_input_hashes["raw survey export"][:12]} | '
    f'pca={_input_hashes["PCA factor loadings"][:12]} | output={OUTPUT_DIR.name}'
)

SOURCE_SNAPSHOT = object_snapshot(raw=raw, retained=retained)


Notebook 3.0-refactor | run 2026-08-04T20:58:10.804693+00:00 | seed 17 | raw=78bd8f48da92 | pca=22b3112a1da4 | output=persona_outputs_refactored


In [4]:
CHARITY_RESOLVER_VERSION = "1.1"
CHARITY_RULE_SOURCE = "clean_charities.py supplied by the user; adapted into pure notebook functions"
CHARITY_WRITEIN_BLOCK = "charity_writein_structure"

CHARITY_MOJIBAKE_MAP = {'‚Äô': "'", '‚Äì': '-', '‚Äî': '-', '‚Äú': '"', '‚Äù': '"', '‚Ä¶': '...', '√†': 'a', '√©': 'e', '√≠': 'i', '√°': 'a', '√ª': 'u', '√±': 'n', '√¥': 'o'}
CHARITY_RULE_SPECS = [{'rule_id': 'rule_000', 'pattern': '^care$', 'canonical': 'CARE', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_001', 'pattern': 'aspca|ascpa|aspaca|apsca|aspc$|^apca$', 'canonical': 'ASPCA', 'cause_area': 'Animal Welfare', 'scope_code': 'nat'}, {'rule_id': 'rule_002', 'pattern': 'humane society|humaine society|human socity|humane organizations', 'canonical': 'Humane Society', 'cause_area': 'Animal Welfare', 'scope_code': 'nat'}, {'rule_id': 'rule_003', 'pattern': '^(none|no|na|n a|not|nope|nothing|nothing much|not sure|idk|not applicable|yesterday|today|good|great|awesome|amazing|fantastic|what|winner|epic donate|relaxing|telling|panda|zombie|blue team|blue|reviews|peers|colleagues|family|jobs|person i know|personal|personal connection|donations|donation|others|other|the only|the only way|the only way i|the only way i could see|m|some|am better|organization|lead by example|hope appeal|give we ll|salna shares|lift up lab|vericalrise|amazing walimat|walimat|american power|american society|follow your heart|crazy pitbull lady|red dead|bed hegad|wate rsee|greyanete|adversary|joenain|spindrift|northwoods|access period|access fund|come and see|yesterday today yesterday|hanover city|hanover|trever|berry queens|koleman|nfag warden|atk stat police|good banks|good oantry|good|hope|love|care|faith)$', 'canonical': '[Non-response / invalid]', 'cause_area': 'Non-response / invalid', 'scope_code': 'junk'}, {'rule_id': 'rule_004', 'pattern': '^(amazon|walmart|target|google|apple|nike|jordan|aetna|usaa|jp morgan|jpmorgan|jp morgan|insurance|consumer reports|bank of america|maf|bbw|pag|qat|zzo|wikipedia|nprgreenhill animal shelter)$', 'canonical': None, 'cause_area': 'Not a charity / for-profit', 'scope_code': 'forprofit'}, {'rule_id': 'rule_005', 'pattern': '(gofundme|go fund me|gofund me|go fund|gofund|go\\-?fund\\-?me)', 'canonical': 'GoFundMe (crowdfunding platform)', 'cause_area': 'Community & General Giving', 'scope_code': 'platform'}, {'rule_id': 'rule_006', 'pattern': 'jude', 'canonical': "St. Jude Children's Research Hospital", 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_007', 'pattern': 'red cross|redcross|red crosd|red crooss|red crodd|red cros|american redcross', 'canonical': 'American Red Cross', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_008', 'pattern': 'salvation|salvage army|slavation|salavation|slavoration|salvatio army|salvations arn|salvation arny', 'canonical': 'Salvation Army', 'cause_area': 'Poverty, Housing & Human Services', 'scope_code': 'nat'}, {'rule_id': 'rule_009', 'pattern': 'goodwill|good will|godwill|gooodwill', 'canonical': 'Goodwill', 'cause_area': 'Poverty, Housing & Human Services', 'scope_code': 'nat'}, {'rule_id': 'rule_010', 'pattern': 'habitat|habit for humanity', 'canonical': 'Habitat for Humanity', 'cause_area': 'Poverty, Housing & Human Services', 'scope_code': 'nat'}, {'rule_id': 'rule_011', 'pattern': 'united way|united ways', 'canonical': 'United Way', 'cause_area': 'Community & General Giving', 'scope_code': 'nat'}, {'rule_id': 'rule_012', 'pattern': 'donors? ?choose|donorschoose|donor s choose|doner s choose|donor ?s? ?choice|donorschoice', 'canonical': 'DonorsChoose', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_013', 'pattern': 'feeding america|feed america|feeing america', 'canonical': 'Feeding America', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_014', 'pattern': 'feed the child|feed the children|feed 4 the poor', 'canonical': 'Feed the Children', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_015', 'pattern': 'save the child|save tbe children|sace the children|savethechildren|save a child', 'canonical': 'Save the Children', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_016', 'pattern': 'unicef|unisef|us fund for unicef', 'canonical': 'UNICEF', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_017', 'pattern': '^unhcr$', 'canonical': 'UNHCR', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_018', 'pattern': '^unrwa$', 'canonical': 'UNRWA', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_019', 'pattern': '^who$|world hunger organization', 'canonical': 'World Health Organization / global health', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_020', 'pattern': 'make ?a ?wish', 'canonical': 'Make-A-Wish Foundation', 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_021', 'pattern': 'without boa?rders?|w o boa?rders?', 'canonical': 'Doctors Without Borders', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_022', 'pattern': 'wounded (warrior|warriors|warrirors|warriers|worriers|worrier|eartuer|warrior project|warriors project)|wonder warrior', 'canonical': 'Wounded Warrior Project', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_023', 'pattern': 'tunnels? to towers?|tunnel 2 towers|tunnel to towert|^t2t$', 'canonical': 'Tunnel to Towers Foundation', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_024', 'pattern': 'kom[ae][nm]|kolman', 'canonical': 'Susan G. Komen', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_025', 'pattern': 'planned parentho', 'canonical': 'Planned Parenthood', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_026', 'pattern': 'american cancer|america cancer|national cancer society|am cancer|am\\. cancer|^acs$', 'canonical': 'American Cancer Society', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_027', 'pattern': 'american heart|heart association|heart assoc|heart assc|heart aoss|am\\. heart|^aha$', 'canonical': 'American Heart Association', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_028', 'pattern': '^rspca$', 'canonical': 'RSPCA', 'cause_area': 'Animal Welfare', 'scope_code': 'nat'}, {'rule_id': 'rule_029', 'pattern': '^spca$', 'canonical': 'SPCA (local societies)', 'cause_area': 'Animal Welfare', 'scope_code': 'local'}, {'rule_id': 'rule_030', 'pattern': '^peta$', 'canonical': 'PETA', 'cause_area': 'Animal Welfare', 'scope_code': 'nat'}, {'rule_id': 'rule_031', 'pattern': 'wwf|world wildlife|worldwildlife|wildlife fund|wildlife foundation', 'canonical': 'World Wildlife Fund (WWF)', 'cause_area': 'Environment & Conservation', 'scope_code': 'nat'}, {'rule_id': 'rule_032', 'pattern': 'greenpeace|green peace', 'canonical': 'Greenpeace', 'cause_area': 'Environment & Conservation', 'scope_code': 'nat'}, {'rule_id': 'rule_033', 'pattern': 'sierra club|serria club', 'canonical': 'Sierra Club', 'cause_area': 'Environment & Conservation', 'scope_code': 'nat'}, {'rule_id': 'rule_034', 'pattern': 'nature conservancy|nature concerbancy', 'canonical': 'The Nature Conservancy', 'cause_area': 'Environment & Conservation', 'scope_code': 'nat'}, {'rule_id': 'rule_035', 'pattern': 'audubon|audobon', 'canonical': 'Audubon Society', 'cause_area': 'Environment & Conservation', 'scope_code': 'nat'}, {'rule_id': 'rule_036', 'pattern': 'rainforest|rails to trails|yellowstone|grand canyon|national wildlife|arbor day|union of concerned scientists|farm sanctuary|sea shepherd|land trust|land connection|wilderness society|conservation|environmental|climate|botanical|arboretum|trees', 'canonical': None, 'cause_area': 'Environment & Conservation', 'scope_code': 'local'}, {'rule_id': 'rule_037', 'pattern': 'shrin|shiners|shribers|shiner hosp', 'canonical': 'Shriners Hospitals for Children', 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_038', 'pattern': 'ronald ?m[cs]?donald|mcdonald house', 'canonical': 'Ronald McDonald House Charities', 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_039', 'pattern': 'miracle network|miriacle network', 'canonical': "Children's Miracle Network", 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_040', 'pattern': 'march of d[im]', 'canonical': 'March of Dimes', 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'nat'}, {'rule_id': 'rule_041', 'pattern': 'children ?s? ?hospital|childrens hospital|children hospital|childrens mercy|phoenix childrens|seattle children|riley|nicklaus|lebonheur|dana far|dana fib|city of hope|memorial sloan|roswell|chop|northwell', 'canonical': None, 'cause_area': "Children's Hospitals & Pediatric Care", 'scope_code': 'local'}, {'rule_id': 'rule_042', 'pattern': '^dav$|disabled american veteran|disabled veteran|disable veteran|disabled vet', 'canonical': 'Disabled American Veterans (DAV)', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_043', 'pattern': '^vfw$|veterans of foreign', 'canonical': 'Veterans of Foreign Wars (VFW)', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_044', 'pattern': 'american legion|amer legion|am legion', 'canonical': 'American Legion', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_045', 'pattern': 'paralyzed vet|paralized vet|^pva$', 'canonical': 'Paralyzed Veterans of America', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_046', 'pattern': 'purple heart', 'canonical': 'Purple Heart / Military Order of the Purple Heart', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_047', 'pattern': 'gary sinise', 'canonical': 'Gary Sinise Foundation', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_048', 'pattern': '^uso$', 'canonical': 'USO', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_049', 'pattern': 'amvets|amv vets', 'canonical': 'AMVETS', 'cause_area': 'Veterans & Military', 'scope_code': 'nat'}, {'rule_id': 'rule_050', 'pattern': 'veteran|vets$|^vets|vet homes|army emergency|homeless vet', 'canonical': None, 'cause_area': 'Veterans & Military', 'scope_code': 'generic'}, {'rule_id': 'rule_051', 'pattern': 'no ?kid ?hungry|no child hungry|nokidhungry', 'canonical': 'No Kid Hungry', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_052', 'pattern': 'world central kitchen|world kitchen|^wck$', 'canonical': 'World Central Kitchen', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_053', 'pattern': 'meals on wheels', 'canonical': 'Meals on Wheels', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_054', 'pattern': 'second harvest', 'canonical': 'Second Harvest', 'cause_area': 'Hunger & Food Security', 'scope_code': 'local'}, {'rule_id': 'rule_055', 'pattern': 'food for the poor|food 4 the poor|food for hungry|food for the hungry', 'canonical': 'Food for the Poor', 'cause_area': 'Hunger & Food Security', 'scope_code': 'nat'}, {'rule_id': 'rule_056', 'pattern': 'city harvest', 'canonical': 'City Harvest', 'cause_area': 'Hunger & Food Security', 'scope_code': 'local'}, {'rule_id': 'rule_057', 'pattern': 'philabundance|forgotte harvest|forgotten harvest|nw harvest|harvesters|houston food bank|oregon food bank|midsouth food bank|greater chicago food|golden harvest|arkansas food|good shepherd food|northern il food|central pa food|los angeles regional food|la food bank|boston food bank|utah food bank|frederick food bank|connecticut foodshare|food depot|food share|food bank of|foodbank|food banks|food bank|food pant|food pantr|foodbanks|soup kitchen|food shelf|food cupboard|food donation|feeding|hunger|feed the homeless|manna|bread', 'canonical': None, 'cause_area': 'Hunger & Food Security', 'scope_code': 'generic'}, {'rule_id': 'rule_058', 'pattern': 'direct relief', 'canonical': 'Direct Relief', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_059', 'pattern': 'americares', 'canonical': 'Americares', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_060', 'pattern': 'team rubicon', 'canonical': 'Team Rubicon', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_061', 'pattern': 'mercy corps', 'canonical': 'Mercy Corps', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_062', 'pattern': 'international rescue committee|^irc$', 'canonical': 'International Rescue Committee (IRC)', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_063', 'pattern': 'convoy of hope', 'canonical': 'Convoy of Hope', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_064', 'pattern': 'world food program', 'canonical': 'World Food Programme', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_065', 'pattern': 'partners in health', 'canonical': 'Partners In Health', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_066', 'pattern': 'operation smile', 'canonical': 'Operation Smile', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_067', 'pattern': 'operation blessing', 'canonical': 'Operation Blessing', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_068', 'pattern': 'care international|^care$', 'canonical': 'CARE', 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'nat'}, {'rule_id': 'rule_069', 'pattern': 'disaster relief|disaster relif|disasters relief|humanitarian', 'canonical': None, 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'generic'}, {'rule_id': 'rule_070', 'pattern': 'catholic charit|cathlic charit', 'canonical': 'Catholic Charities', 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_071', 'pattern': 'compassion international|^compassion$', 'canonical': 'Compassion International', 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_072', 'pattern': 'world vision|worldision', 'canonical': 'World Vision', 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_073', 'pattern': 'samaritan', 'canonical': "Samaritan's Purse", 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_074', 'pattern': 'latter ?day saints|latter-day|church of jesus christ|^lds$', 'canonical': 'Church of Jesus Christ of Latter-day Saints', 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_075', 'pattern': '^ifcj$|fellowship of christians and jews|christians and jews', 'canonical': 'Intl Fellowship of Christians & Jews', 'cause_area': 'Faith & Religious', 'scope_code': 'nat'}, {'rule_id': 'rule_076', 'pattern': 'td jakes|joel osteen|joyce meyer|david jeremiah|andrew womack|in touch|prison fellowship|thru the bible|ministr|missionar|missions|mission asset|gospel mission|rescue mission|garden mission|maryknoll|franciscan|salesian|st vincent|knights of columbus|umcor|jehovah|synagogue|temple|mosque|zakat|islamic relief|kabbalah|marble collegiate|apostolic|baptist|methodist|lutheran|presbyterian|episcopal|catholic church|seraphic|abwe|world relief|water with blessings|show hope', 'canonical': None, 'cause_area': 'Faith & Religious', 'scope_code': 'local'}, {'rule_id': 'rule_077', 'pattern': '^church|churches|my church|local church|our church|the church|catholic org|religious|religion|diocese|parish|cathedral|new life church|cowboy church|bayside church', 'canonical': 'Church / religious congregation (unspecified)', 'cause_area': 'Faith & Religious', 'scope_code': 'generic'}, {'rule_id': 'rule_078', 'pattern': 'boys? ?and ?girls club|boys girls club|boy s and girls clu|boys and girls clubs', 'canonical': 'Boys & Girls Clubs', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_079', 'pattern': 'big brother', 'canonical': 'Big Brothers Big Sisters', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_080', 'pattern': '^ymca$', 'canonical': 'YMCA', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_081', 'pattern': 'boys? ?town', 'canonical': 'Boys Town', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_082', 'pattern': 'girl scout|girls scout|brownies', 'canonical': 'Girl Scouts', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_083', 'pattern': 'boy scout|^bsa$', 'canonical': 'Boy Scouts of America', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_084', 'pattern': 'special olympic', 'canonical': 'Special Olympics', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_085', 'pattern': 'indian school|indian college|st joseph.*indian|st labre|st labored|labre', 'canonical': "St. Joseph's / St. Labre Indian Schools", 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_086', 'pattern': 'child fund|childfund|cradles to crayons|reading is fundamental|literacy|scouting|4h|head start|junior achievement|club 21|casa$|^casa|court appointed', 'canonical': None, 'cause_area': 'Children, Youth & Education', 'scope_code': 'local'}, {'rule_id': 'rule_087', 'pattern': '\\bschool|college|colleges|universit|univ of|high school|pto|pta|education|youth|scholarship|band|music boosters|athletics|sports team|tuition|classroom|girls who code|girls with anxiety|day care|daycare|children s education|children health|^children$|^kids$|^child$', 'canonical': None, 'cause_area': 'Children, Youth & Education', 'scope_code': 'generic'}, {'rule_id': 'rule_088', 'pattern': 'oxfam', 'canonical': 'Oxfam', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_089', 'pattern': '^kiva$', 'canonical': 'Kiva', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_090', 'pattern': 'givedirectly|give directly', 'canonical': 'GiveDirectly', 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_091', 'pattern': 'one acre fund|living goods|malaria consortium|iodine global|plan int|plan intl|international health|global poverty|micro business|water well|mercy ships|mercy chef|heifer|hefer project|sanku', 'canonical': None, 'cause_area': 'International Aid & Development', 'scope_code': 'nat'}, {'rule_id': 'rule_092', 'pattern': '^aclu$|american civil liberties', 'canonical': 'ACLU', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_093', 'pattern': 'naacp', 'canonical': 'NAACP', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_094', 'pattern': 'splc|southern pove?rty law', 'canonical': 'Southern Poverty Law Center', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_095', 'pattern': 'human rights campaign', 'canonical': 'Human Rights Campaign', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_096', 'pattern': 'trevor project|trevor', 'canonical': 'The Trevor Project', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_097', 'pattern': 'equal justice', 'canonical': 'Equal Justice Initiative', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_098', 'pattern': 'innocence project', 'canonical': 'Innocence Project', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_099', 'pattern': 'rainn|rape abuse', 'canonical': 'RAINN', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_100', 'pattern': '^adl$|anti defamation', 'canonical': 'Anti-Defamation League', 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_101', 'pattern': 'naral|abortion fund|access to healthcare|emily s list|loveland foundation', 'canonical': None, 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'nat'}, {'rule_id': 'rule_102', 'pattern': 'lgbt|lgbtq|human rights|womens? (rights|shelter|health|empowerment|abuse|in transition)|women s|social justice|civil libert|domestic violence|victims of torture|human rights first', 'canonical': None, 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'generic'}, {'rule_id': 'rule_103', 'pattern': 'als association|^als$', 'canonical': 'ALS Association', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_104', 'pattern': 'multiple sclerosis|ms society|multiple sch?erosis|multiple sckerosis|national ms|^ms$', 'canonical': 'National MS Society', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_105', 'pattern': 'alzheimer|alz$|^alz|ahlzheimers|alzehmeiers|allzheimers|alzheimrrs', 'canonical': "Alzheimer's Association", 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_106', 'pattern': 'lupus', 'canonical': 'Lupus Foundation', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_107', 'pattern': 'cystic fibrosis|^cf foundation', 'canonical': 'Cystic Fibrosis Foundation', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_108', 'pattern': 'leukemia|lymphoma', 'canonical': 'Leukemia & Lymphoma Society', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_109', 'pattern': 'parkinson', 'canonical': "Parkinson's Foundation", 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_110', 'pattern': 'epilepsy', 'canonical': 'Epilepsy Foundation', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_111', 'pattern': 'crohn', 'canonical': "Crohn's & Colitis Foundation", 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_112', 'pattern': 'stand up to cancer', 'canonical': 'Stand Up To Cancer', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_113', 'pattern': 'relay for life', 'canonical': 'Relay for Life', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_114', 'pattern': 'jimmy fund', 'canonical': 'Jimmy Fund', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_115', 'pattern': 'v foundation', 'canonical': 'The V Foundation', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_116', 'pattern': 'smile train', 'canonical': 'Smile Train', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'nat'}, {'rule_id': 'rule_117', 'pattern': 'hospice|hospicare', 'canonical': None, 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'local'}, {'rule_id': 'rule_118', 'pattern': 'cancer|hiv|aids|diabetes|autism|blood|plasma|lung|kidney|hearing|sight|blind|mental health|suicide|suicode|health|medical|hospital|clinic|breast|prostate|colon|sclerosis|nami|pcrf|ms walk|disease|research|^heart$', 'canonical': None, 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'generic'}, {'rule_id': 'rule_119', 'pattern': '\\bnpr\\b|public radio|public television|^pbs$|mtpr|kera|kqed|kycc|air1|mpt', 'canonical': 'Public Radio / PBS / NPR', 'cause_area': 'Arts, Culture & Media', 'scope_code': 'nat'}, {'rule_id': 'rule_120', 'pattern': 'museum|symphony|opera|ballet|theater|theatre|orchestra|gallery|arts|cultural trust|library|libraries|historical society|holocaust museum|piano cleveland', 'canonical': None, 'cause_area': 'Arts, Culture & Media', 'scope_code': 'local'}, {'rule_id': 'rule_121', 'pattern': 'trump|^rnc$|republican national|^nra$|prageru|convention of states', 'canonical': 'Conservative/Republican political', 'cause_area': 'Political', 'scope_code': 'nat'}, {'rule_id': 'rule_122', 'pattern': '^dnc$|democrat|democratic party|democrats|act blue|actblue|clinton foundation', 'canonical': 'Democratic/progressive political', 'cause_area': 'Political', 'scope_code': 'nat'}, {'rule_id': 'rule_123', 'pattern': 'political|politics|campaign', 'canonical': None, 'cause_area': 'Political', 'scope_code': 'generic'}, {'rule_id': 'rule_124', 'pattern': 'rotary|^lions$|lions club|lions international|kiwanis|elks|optimist club|community foundation|united foundation|urban league|chamber of commerce', 'canonical': None, 'cause_area': 'Community & General Giving', 'scope_code': 'local'}, {'rule_id': 'rule_125', 'pattern': 'local charit|local charities|^community$|community first|community event|community blood|community improvement|community initiatives|donor$|donors$', 'canonical': None, 'cause_area': 'Community & General Giving', 'scope_code': 'generic'}, {'rule_id': 'rule_126', 'pattern': 'animal|pet|dog|cat|kitty|puppy|feline|canine|equine|horse|paws|rescue.*animal|animal.*rescue|shelter.*animal|humane|zoo|birds|elephant|wildlife warriors|best friends|friends of animals|friends for life|friends of ferals|fluff|foxes cradle', 'canonical': None, 'cause_area': 'Animal Welfare', 'scope_code': 'local'}, {'rule_id': 'rule_127', 'pattern': 'toys? ?(for|4) ?tots|toys for toys', 'canonical': 'Toys for Tots', 'cause_area': 'Children, Youth & Education', 'scope_code': 'nat'}, {'rule_id': 'rule_128', 'pattern': '^aarp$', 'canonical': 'AARP', 'cause_area': 'Community & General Giving', 'scope_code': 'nat'}, {'rule_id': 'rule_129', 'pattern': '^va$|veterans affairs', 'canonical': 'Veterans / VA (unspecified)', 'cause_area': 'Veterans & Military', 'scope_code': 'generic'}, {'rule_id': 'rule_130', 'pattern': '^aa$', 'canonical': 'Alcoholics Anonymous / recovery', 'cause_area': 'Disease & Medical Research/Advocacy', 'scope_code': 'generic'}, {'rule_id': 'rule_131', 'pattern': 'police|fire ?fighter|firefighter|first responder|police and fire|^fop$', 'canonical': 'Police / fire / first responders', 'cause_area': 'Community & General Giving', 'scope_code': 'generic'}, {'rule_id': 'rule_132', 'pattern': 'dream center|haven of|city of hope mission|rescue mission', 'canonical': None, 'cause_area': 'Faith & Religious', 'scope_code': 'local'}, {'rule_id': 'rule_133', 'pattern': 'homeless|shelter|thrift|clothing|clothes|women in transition|transitional|st vincent de paul|st vincent depaul|little sisters of the poor|goodwill|salvation|community service|community aide|community care|community support|helping hand|people helping people|dress for success|dress for sucess|refuge|housing|homes on wheels|rebuilding|opportunity center|caring|families in need|people against heroine|addiction|recovery|foster', 'canonical': None, 'cause_area': 'Poverty, Housing & Human Services', 'scope_code': 'generic'}, {'rule_id': 'rule_134', 'pattern': '^military$', 'canonical': None, 'cause_area': 'Veterans & Military', 'scope_code': 'generic'}, {'rule_id': 'rule_135', 'pattern': '^relief$|^disasters? relief$', 'canonical': None, 'cause_area': 'Disaster & Humanitarian Relief', 'scope_code': 'generic'}, {'rule_id': 'rule_136', 'pattern': '^arts?$', 'canonical': None, 'cause_area': 'Arts, Culture & Media', 'scope_code': 'generic'}, {'rule_id': 'rule_137', 'pattern': '^sick (children|kids)$|^orphan$|^disability$', 'canonical': None, 'cause_area': 'Children, Youth & Education', 'scope_code': 'generic'}, {'rule_id': 'rule_138', 'pattern': '^women groups?$|^womens?$', 'canonical': None, 'cause_area': 'Civil Rights, Advocacy & Legal', 'scope_code': 'generic'}]
CHARITY_RULES = [
    (re.compile(spec["pattern"]), spec)
    for spec in CHARITY_RULE_SPECS
]

CHARITY_SCOPE_MAP = {
    "nat": ("national_or_well_known_org", "national_or_well_known"),
    "local": ("local_or_oneoff_org", "local_or_regional"),
    "generic": ("generic_cause", "unspecified"),
    "platform": ("crowdfunding_platform", "not_applicable"),
    "forprofit": ("for_profit_not_charity", "not_applicable"),
    "junk": ("nonresponse_invalid", "not_applicable"),
}


def charity_fix_text(value: Any) -> str:
    text = "" if pd.isna(value) else str(value)
    for bad, good in CHARITY_MOJIBAKE_MAP.items():
        text = text.replace(bad, good)
    return text.strip().strip("`").strip(".").strip()


def charity_key(value: Any) -> str:
    text = charity_fix_text(value).lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def charity_resolution_dimensions(scope_code: str, matched: bool) -> tuple[str, str]:
    if not matched:
        return "unresolved_specific", "unspecified"
    return CHARITY_SCOPE_MAP[scope_code]


def resolve_charity_mention(raw_text: Any) -> dict[str, Any]:
    cleaned = charity_fix_text(raw_text)
    normalized = charity_key(cleaned)
    if not normalized:
        return {
            "cleaned_text": cleaned, "normalized_key": normalized, "canonical_rule": None,
            "cause_area": "Non-response / invalid", "entity_kind": "nonresponse_invalid",
            "geographic_scope": "not_applicable", "matched_rule": None,
            "resolution_method": "empty", "confidence": 1.0, "review_flag": False,
        }

    for pattern, spec in CHARITY_RULES:
        if pattern.search(normalized):
            entity_kind, geographic_scope = charity_resolution_dimensions(spec["scope_code"], matched=True)
            has_canonical = spec["canonical"] is not None
            confidence = 0.98 if has_canonical else (0.95 if spec["scope_code"] in {"junk", "forprofit", "platform"} else 0.75)
            review_flag = entity_kind not in {"nonresponse_invalid", "for_profit_not_charity"} and not has_canonical
            return {
                "cleaned_text": cleaned,
                "normalized_key": normalized,
                "canonical_rule": spec["canonical"],
                "cause_area": spec["cause_area"],
                "entity_kind": entity_kind,
                "geographic_scope": geographic_scope,
                "matched_rule": spec["rule_id"],
                "resolution_method": "canonical_rule" if has_canonical else "heuristic_rule",
                "confidence": confidence,
                "review_flag": review_flag,
            }

    return {
        "cleaned_text": cleaned,
        "normalized_key": normalized,
        "canonical_rule": None,
        "cause_area": "Other / unclassified",
        "entity_kind": "unresolved_specific",
        "geographic_scope": "unspecified",
        "matched_rule": None,
        "resolution_method": "unmatched",
        "confidence": 0.35,
        "review_flag": True,
    }


ORG_WRITEIN_QUESTION = (
    "To what organizations and/or causes do you typically make charitable donations? "
    "If you are comfortable sharing, please list the specific charities or organizations below. "
    "If you do not feel comfortable sharing, please put N/A in the first text line."
)
ORG_WRITEIN_COLUMNS = tuple(f"{rank}. - {ORG_WRITEIN_QUESTION}" for rank in range(1, 6))


def extract_charity_mentions(frame: pd.DataFrame) -> pd.DataFrame:
    missing = set(ORG_WRITEIN_COLUMNS).difference(frame.columns)
    if missing:
        raise KeyError(f"Missing charity write-in columns: {sorted(missing)}")
    mentions = frame[[ID_COL, SOURCE_COL, *ORG_WRITEIN_COLUMNS]].melt(
        id_vars=[ID_COL, SOURCE_COL],
        value_vars=list(ORG_WRITEIN_COLUMNS),
        var_name="mention_slot",
        value_name="raw_text",
    )
    mentions["mention_rank"] = mentions["mention_slot"].str.extract(r"^(\d+)").astype("Int64")
    mentions = mentions[mentions["raw_text"].notna()].copy()
    resolved = pd.DataFrame(mentions["raw_text"].map(resolve_charity_mention).tolist(), index=mentions.index)
    mentions = pd.concat([mentions, resolved], axis=1)
    mentions = mentions[mentions["normalized_key"].ne("")].copy()

    mentions["entity_group_key"] = np.where(
        mentions["canonical_rule"].notna(),
        "CANON::" + mentions["canonical_rule"].astype(str),
        "RAW::" + mentions["normalized_key"],
    )
    display_names = (
        mentions.groupby("entity_group_key")["cleaned_text"]
        .agg(lambda values: values.value_counts().index[0])
        .to_dict()
    )
    mentions["canonical_entity"] = mentions["canonical_rule"].fillna(
        mentions["entity_group_key"].map(display_names)
    )
    mentions["is_charitable_mention"] = ~mentions["entity_kind"].isin(
        ["nonresponse_invalid", "for_profit_not_charity"]
    )
    mentions["review_reason"] = np.select(
        [
            mentions["resolution_method"].eq("unmatched"),
            mentions["resolution_method"].eq("heuristic_rule"),
        ],
        ["unmatched long tail", "broad keyword heuristic"],
        default="",
    )
    return mentions.sort_values([ID_COL, "mention_rank"]).reset_index(drop=True)


def build_charity_entity_dictionary(mentions: pd.DataFrame) -> pd.DataFrame:
    usable = mentions[mentions["is_charitable_mention"]].copy()
    rows = []
    for entity, group in usable.groupby("canonical_entity", sort=False):
        rows.append({
            "canonical_entity": entity,
            "cause_area": group["cause_area"].mode().iat[0],
            "entity_kind": group["entity_kind"].mode().iat[0],
            "geographic_scope": group["geographic_scope"].mode().iat[0],
            "mention_count": len(group),
            "respondent_count": group[ID_COL].nunique(),
            "n_raw_variants": group["raw_text"].nunique(),
            "example_raw_variants": " | ".join(group["raw_text"].value_counts().head(6).index.astype(str)),
            "minimum_confidence": group["confidence"].min(),
            "review_flag": bool(group["review_flag"].any()),
        })
    return pd.DataFrame(rows).sort_values(["mention_count", "canonical_entity"], ascending=[False, True]).reset_index(drop=True)


def build_charity_review_queue(mentions: pd.DataFrame) -> pd.DataFrame:
    review = mentions[mentions["is_charitable_mention"]].copy()
    rows = []
    for normalized, group in review.groupby("normalized_key", sort=False):
        needs_review = bool(group["review_flag"].any())
        rows.append({
            "normalized_key": normalized,
            "current_canonical_entity": group["canonical_entity"].mode().iat[0],
            "current_cause_area": group["cause_area"].mode().iat[0],
            "current_entity_kind": group["entity_kind"].mode().iat[0],
            "current_geographic_scope": group["geographic_scope"].mode().iat[0],
            "matched_rule": group["matched_rule"].dropna().astype(str).mode().iat[0] if group["matched_rule"].notna().any() else None,
            "minimum_confidence": group["confidence"].min(),
            "mention_count": len(group),
            "respondent_count": group[ID_COL].nunique(),
            "raw_examples": " | ".join(group["raw_text"].value_counts().head(6).index.astype(str)),
            "needs_review": needs_review,
            "review_priority": "high" if needs_review and group[ID_COL].nunique() >= 3 else ("standard" if needs_review else "resolved"),
            "review_reason": " | ".join(sorted(set(filter(None, group["review_reason"].astype(str))))),
        })
    return pd.DataFrame(rows).sort_values(
        ["needs_review", "respondent_count", "mention_count"], ascending=[False, False, False]
    ).reset_index(drop=True)


def build_charity_respondent_features(frame: pd.DataFrame, mentions: pd.DataFrame) -> pd.DataFrame:
    base = frame[[ID_COL, SOURCE_COL]].drop_duplicates().copy()
    usable = mentions[mentions["is_charitable_mention"]].copy()
    grouped = usable.groupby(ID_COL)
    summary = grouped.agg(
        writein_valid_mention_count=("canonical_entity", "size"),
        writein_distinct_entity_count=("canonical_entity", "nunique"),
        writein_distinct_cause_count=("cause_area", "nunique"),
    )
    summary["writein_share_local"] = grouped["geographic_scope"].apply(lambda s: s.eq("local_or_regional").mean())
    summary["writein_share_national"] = grouped["geographic_scope"].apply(lambda s: s.eq("national_or_well_known").mean())
    summary["writein_share_specific_entity"] = grouped["entity_kind"].apply(
        lambda s: s.isin(["national_or_well_known_org", "local_or_oneoff_org", "unresolved_specific"]).mean()
    )
    summary["writein_share_generic_cause"] = grouped["entity_kind"].apply(lambda s: s.eq("generic_cause").mean())
    summary["writein_crowdfunding_indicator"] = grouped["entity_kind"].apply(lambda s: float(s.eq("crowdfunding_platform").any()))
    first = usable.sort_values([ID_COL, "mention_rank"]).drop_duplicates(ID_COL).set_index(ID_COL)
    summary["writein_first_mention_local"] = first["geographic_scope"].eq("local_or_regional").astype(float)
    result = base.merge(summary, left_on=ID_COL, right_index=True, how="left", validate="one_to_one")
    count_columns = ["writein_valid_mention_count", "writein_distinct_entity_count", "writein_distinct_cause_count"]
    indicator_columns = ["writein_crowdfunding_indicator"]
    result[count_columns + indicator_columns] = result[count_columns + indicator_columns].fillna(0)
    result["writein_has_usable_mention"] = result["writein_valid_mention_count"].gt(0).astype(float)
    return result


CHARITY_MENTIONS = extract_charity_mentions(retained)

CHARITY_ENTITY_DICTIONARY = build_charity_entity_dictionary(CHARITY_MENTIONS)
CHARITY_REVIEW_QUEUE = build_charity_review_queue(CHARITY_MENTIONS)
CHARITY_RESPONDENT_FEATURES = build_charity_respondent_features(retained, CHARITY_MENTIONS)

# Only non-cause structure is eligible for clustering. Mention count remains a guardrail/profile field.
CHARITY_STRUCTURE_CLUSTER_FEATURES = {
    "writein_distinct_cause_count": 5.0,
    "writein_share_local": 1.0,
    "writein_share_specific_entity": 1.0,
    "writein_share_generic_cause": 1.0,
    "writein_crowdfunding_indicator": 1.0,
    "writein_first_mention_local": 1.0,
}

# Common exact entities are used only as a cluster-dominance guardrail and later profile output.
_common = CHARITY_MENTIONS[
    CHARITY_MENTIONS["is_charitable_mention"] & CHARITY_MENTIONS[SOURCE_COL].eq("Panel")
].drop_duplicates([ID_COL, "canonical_entity"])
_common_counts = _common.groupby("canonical_entity")[ID_COL].nunique()
_common_entities = _common_counts[_common_counts >= 10].index
CHARITY_COMMON_ENTITY_PRESENCE = (
    _common[_common["canonical_entity"].isin(_common_entities)]
    .assign(value=1)
    .pivot_table(index=ID_COL, columns="canonical_entity", values="value", fill_value=0)
)


In [5]:
@dataclass(frozen=True)
class DistanceUnit:
    name: str
    block: str
    kind: str
    columns: tuple[str, ...]
    mapping: Mapping[str, float] | None = None
    ordered_range: float | None = None
    availability_column: str | None = None
    availability_selected_when: bool | None = None
    log1p: bool = False
    cap_quantile: float = 0.99
    notes: str = ''


def make_likert_columns(items: Sequence[str], question: str) -> tuple[str, ...]:
    return tuple(likert_col(item, question) for item in items)


def make_option_columns(question: str, options: Sequence[str]) -> tuple[str, ...]:
    return tuple(option_col(question, option) for option in options)

impact_options = ('My local community', 'My state/region', 'My country', 'Other countries/International', 'No preference')

core_motivation_options = (
    'Personal connection to the cause or issue',
    'Desire to make a positive impact in the community',
    'Influence from family, friends, or peers',
    'Tax benefits or financial incentives',
    'Belief in the mission and values of the organization',
    'Trust in the effectiveness and transparency of the organization',
    'Corporate matching programs through my employer',
    'Opportunities for personal or professional recognition',
    'Religious or spiritual beliefs',
    "Desire to honor someone's memory or legacy",
    'Participation in events or fundraisers',
    'Regular habit or tradition of giving',
    'Someone from the charity asks for a donation',
    'I know someone directly involved with the charity',
    'The charity/organization directly impacts a friend or family member',
)
trigger_motivation_options = (
    'Emotional response to a specific event or campaign',
    'Information about the urgent need for support',
    'Social media campaigns or celebrity endorsements',
)

giving_attitude_items = (
    'I make donations to organizations/causes that align with my personal values',
    'I believe charitable donations make a significant impact on the community',
    'I prefer to donate to local charities rather than national or international ones',
    'I feel a sense of personal satisfaction when I make a charitable donation',
    'I regularly set aside a portion of my income for charitable donations',
    'I believe material impact is more important than advocacy or awareness-building',
    'I donate to charity out of a moral/religious obligation',
)

importance_items = (
    "Effectiveness and impact of the organization's programs",
    'Transparency and accountability of the organization',
    'Efficiency in using donations (low administrative costs)',
    'Recommendations or endorsements from trusted sources',
    'Availability of tax deductions for donations',
    'Personal connection to the organization/cause',
    "Diversity and inclusivity in the organization's practices and leadership",
    'Recognition or acknowledgment of donors',
    'Presence of matching gift opportunities',
    'Security and ease of the donation process',
    'Opportunities for personal involvement or volunteering with the organization',
    'My employer matching the donation',
)

research_source_options = (
    'Organization websites',
    'Charity review websites',
    'Blogs',
    'Catalogs or magazines',
    'Television advertisements',
    'Television programs',
    'Recommendations from family/friends',
    'News articles',
    'Online reviews',
    'Visits to the organization',
    'Search engines (e.g., Google, Bing)',
    'ChatGPT',
)
research_info_options = (
    'Reviews/Ratings/Testimonials',
    'Reliability',
    'Donation center locations',
    'Community services',
    'Internal structure of the organization',
    'Communities the organization serves',
    'Financial reporting/fiscal responsibility',
    'Environmental stewardship',
    'Volunteer opportunities and ways to get involved',
    'Specific projects or initiatives currently underway',
    'Awards and recognitions received by the organization',
    'Methods of communication and donor engagement',
    'Donation process',
)
challenge_options = (
    'Difficulty in verifying the legitimacy of the charity or organization',
    'Concerns about how my donation will be used',
    "Lack of transparency in the charity's financial reporting",
    'Difficulty deciding which organization/cause or charity to support',
    'Uncertainties about the tax implications of my donation',
    'Lack of feedback or results from previous donations',
    'The process of making a donation (e.g., payment methods, paperwork)',
    'Economic uncertainties that impact my ability to donate',
    'Difficulty finding charities that align with my values and interests',
    'Peer influence or social pressure',
    'Uncertainty about the effectiveness of my donation in making a real difference',
)
trust_attitude_items = (
    'I research an organization thoroughly before making a donation',
    'I tend to stick to organizations that I trust and prefer and do not seek to find new ones for potential donations',
)
trust_message_items = (
    'Understanding the specific impact of your donation (e.g., how funds are used, success stories)',
    "Information about the charity's transparency and financial accountability",
    'A clear and compelling mission statement or vision from the charity',
    'Messages that align with your personal values and belief',
)

trigger_discovery_options = (
    'Stories or testimonials',
    'News articles',
    'Documentaries',
    'Social media campaigns or influencers promoting a cause',
    'Celebrity endorsements in a charity',
    'Events or fundraisers related to a cause',
)
trigger_research_options = ('Facebook', 'Pinterest', 'Instagram', 'TikTok', 'YouTube', 'Twitch', 'Kick')
trigger_message_items = (
    'Seeing personal stories or testimonials from those who have benefited from the charity',
    'Messages that emphasize the urgency of the situation or need',
    'Celebrity endorsements or partnerships promoting the charity',
    'Recommendations or endorsements from trusted sources (e.g., friends, family)',
    'Opportunities to see or participate in the charity\'s work firsthand (e.g., site visits, volunteer opportunities)',
    'Information about matching donation opportunities (e.g., employer match, challenge grants)',
    'Appeals that include a sense of community involvement or local impact',
    'Messages that highlight the ease and convenience of the donation process',
)
trigger_attitude_items = (
    'I am influenced by social media campaigns to make donations',
    'I am more likely to donate during times of crisis or disaster',
    'I am more likely to donate to a cause if I know someone personally affected by it',
    'I am more likely to donate to charities that my employer matches contributions to',
)
ad_response_options = (
    'Placement in TV shows and movies',
    'Ads in between TV shows',
    'Ads/online banners on websites',
    'Ads placed in the middle of streaming TV shows and movies',
    "An organization's website or webpage",
    'Ads placed at the beginning of streaming TV shows and movies',
    'Radio ads',
    'Ads embedded in social media apps',
    'Ads embedded in news apps',
    'News articles',
)

education_format_options = (
    'Donations to higher education institutions (e.g., universities, colleges)',
    'Contributions to scholarship funds',
    "Direct donation to specific teachers' classrooms",
    'Donations to after-school or extracurricular programs',
    'Contributions to adult education or vocational training programs',
    'I would not be likely to make any education-related donations',
)

ACTIVE_UNITS = [
    DistanceUnit('donation_frequency', 'giving_commitment', 'ordinal', (FREQUENCY_COL,), FREQUENCY_MAP, 4),
    DistanceUnit('donation_amount', 'giving_commitment', 'ordinal', (AMOUNT_COL,), AMOUNT_MAP, 7),
    DistanceUnit('organizations_per_year', 'giving_commitment', 'numeric', (ORG_COUNT_COL,), log1p=True),
    DistanceUnit('donations_per_year', 'giving_commitment', 'numeric', (DONATION_COUNT_COL,), log1p=True),
    DistanceUnit('decision_state', 'giving_commitment', 'nominal', (DECISION_COL,)),

    DistanceUnit('core_motivations', 'giving_job_impact', 'multiselect', make_option_columns(Q_MOTIVATIONS, core_motivation_options)),
    DistanceUnit('impact_scale_preferences', 'giving_job_impact', 'multiselect', make_option_columns(Q_IMPACT_SCALE, impact_options), notes='Instrument says single choice; export contains multiple selections, so model observed selections with Jaccard.'),
    DistanceUnit('giving_attitudes', 'giving_job_impact', 'likert_battery', make_likert_columns(giving_attitude_items, Q_CHARITY_ATTITUDES), AGREE_MAP, 4),

    DistanceUnit('organization_selection_attributes', 'trust_research_friction', 'likert_battery', make_likert_columns(importance_items, Q_IMPORTANCE), IMPORTANT_MAP, 4),
    DistanceUnit('research_sources', 'trust_research_friction', 'multiselect', make_option_columns(Q_RESEARCH_SOURCES, research_source_options), availability_column=option_col(Q_RESEARCH_SOURCES, 'I do not conduct research before making a charitable donation'), availability_selected_when=False),
    DistanceUnit('research_information', 'trust_research_friction', 'multiselect', make_option_columns(Q_RESEARCH_INFO, research_info_options), availability_column=option_col(Q_RESEARCH_SOURCES, 'I do not conduct research before making a charitable donation'), availability_selected_when=False),
    DistanceUnit('donation_challenges', 'trust_research_friction', 'multiselect', make_option_columns(Q_CHALLENGES, challenge_options)),
    DistanceUnit('research_and_incumbent_trust', 'trust_research_friction', 'likert_battery', make_likert_columns(trust_attitude_items, Q_CHARITY_ATTITUDES), AGREE_MAP, 4),
    DistanceUnit('trust_messages', 'trust_research_friction', 'likert_battery', make_likert_columns(trust_message_items, Q_MESSAGING), INFLUENTIAL_MAP, 4),

    DistanceUnit('trigger_motivations', 'trigger_channel', 'multiselect', make_option_columns(Q_MOTIVATIONS, trigger_motivation_options)),
    DistanceUnit('trigger_discovery', 'trigger_channel', 'multiselect', make_option_columns(Q_DISCOVERY, trigger_discovery_options)),
    DistanceUnit('social_research_channels', 'trigger_channel', 'multiselect', make_option_columns(Q_RESEARCH_SOURCES, trigger_research_options), availability_column=option_col(Q_RESEARCH_SOURCES, 'I do not conduct research before making a charitable donation'), availability_selected_when=False),
    DistanceUnit('trigger_messages', 'trigger_channel', 'likert_battery', make_likert_columns(trigger_message_items, Q_MESSAGING), INFLUENTIAL_MAP, 4),
    DistanceUnit('trigger_attitudes', 'trigger_channel', 'likert_battery', make_likert_columns(trigger_attitude_items, Q_CHARITY_ATTITUDES), AGREE_MAP, 4),
    DistanceUnit('advertising_response', 'trigger_channel', 'multiselect', make_option_columns(Q_AD_RESPONSE, ad_response_options)),

    DistanceUnit('current_education_giver', 'education_receptivity', 'binary_selected', (option_col(Q_CURRENT_CAUSES, 'Educational-focused groups'),)),
    DistanceUnit('education_openness_non_giver', 'education_receptivity', 'ordinal', (likert_col('Educational-focused groups', Q_OPENNESS),), OPEN_MAP, 4, availability_column=option_col(Q_CURRENT_CAUSES, 'Educational-focused groups'), availability_selected_when=False),
    DistanceUnit('education_giving_format', 'education_receptivity', 'multiselect', make_option_columns(Q_EDU_FORMATS, education_format_options)),
    DistanceUnit('teacher_community_discovery', 'education_receptivity', 'binary_selected', (option_col(Q_DISCOVERY, 'Direct requests from community figures (e.g., teachers, organizers)'),)),
    DistanceUnit('current_children_youth_giver', 'education_receptivity', 'binary_selected', (option_col(Q_CURRENT_CAUSES, "Children's charities and youth programs"),)),
]

@dataclass(frozen=True)
class SegmentExperiment:
    """Declarative clustering experiment. Edit data, not engine code."""

    name: str
    include_blocks: tuple[str, ...]
    include_units: tuple[str, ...] | None = None
    exclude_units: tuple[str, ...] = ()
    block_weights: Mapping[str, float] = field(default_factory=dict)
    unit_weights: Mapping[str, float] = field(default_factory=dict)
    response_style_adjusted: bool = False
    k_values: tuple[int, ...] = (3, 4, 5, 6)
    notes: str = ''

    def as_dict(self) -> dict[str, Any]:
        return {
            'name': self.name,
            'include_blocks': list(self.include_blocks),
            'include_units': None if self.include_units is None else list(self.include_units),
            'exclude_units': list(self.exclude_units),
            'block_weights': dict(self.block_weights),
            'unit_weights': dict(self.unit_weights),
            'response_style_adjusted': self.response_style_adjusted,
            'k_values': list(self.k_values),
            'notes': self.notes,
        }


UNIT_LOOKUP = {unit.name: unit for unit in ACTIVE_UNITS}
assert len(UNIT_LOOKUP) == len(ACTIVE_UNITS)


ACTIVE_UNIT_NAMES = {unit.name for unit in ACTIVE_UNITS}
for unit in ACTIVE_UNITS:
    missing = set(unit.columns).difference(raw.columns)
    assert not missing, f'{unit.name}: missing columns={sorted(missing)}'
    assert unit.block in BLOCK_NAMES
assert len(UNIT_LOOKUP) == len(ACTIVE_UNITS)


In [6]:
def availability_vector(frame: pd.DataFrame, unit: DistanceUnit) -> np.ndarray:
    if unit.availability_column is None:
        return np.ones(len(frame), dtype=bool)
    selected_flag = frame[unit.availability_column].notna().to_numpy()
    return selected_flag if unit.availability_selected_when else ~selected_flag


def pair_availability(row_available: np.ndarray) -> np.ndarray:
    return row_available[:, None] & row_available[None, :]


def pairwise_single_numeric(values: np.ndarray, value_range: float, row_available: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    valid = row_available & np.isfinite(values)
    available = pair_availability(valid)
    distance = np.abs(values[:, None] - values[None, :]) / value_range
    distance[~available] = 0.0
    return distance.astype(np.float32), available


def pairwise_nominal(values: np.ndarray, row_available: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    valid = row_available & pd.notna(values)
    available = pair_availability(valid)
    distance = (values[:, None] != values[None, :]).astype(np.float32)
    distance[~available] = 0.0
    return distance, available


def pairwise_multiselect(binary: np.ndarray, row_available: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Jaccard uses selected options only; shared non-selections never increase similarity.
    binary = binary.astype(np.float32)
    intersection = binary @ binary.T
    selected_count = binary.sum(axis=1)
    union = selected_count[:, None] + selected_count[None, :] - intersection
    distance = np.divide(
        union - intersection,
        union,
        out=np.zeros_like(union, dtype=np.float32),
        where=union > 0,
    )
    available = pair_availability(row_available)
    distance[~available] = 0.0
    return distance.astype(np.float32), available


def pairwise_likert_battery(
    values: np.ndarray,
    scale_range: float,
    row_available: np.ndarray,
    response_style_adjusted: bool,
) -> tuple[np.ndarray, np.ndarray]:
    n, m = values.shape
    item_sum = np.zeros((n, n), dtype=np.float32)
    item_count = np.zeros((n, n), dtype=np.int16)
    for j in range(m):
        x = values[:, j]
        valid = row_available & np.isfinite(x)
        available_j = pair_availability(valid)
        d = np.abs(x[:, None] - x[None, :]) / scale_range
        item_sum += np.where(available_j, d, 0).astype(np.float32)
        item_count += available_j.astype(np.int16)
    raw_distance = np.divide(item_sum, item_count, out=np.zeros_like(item_sum), where=item_count > 0)
    available = item_count > 0
    if not response_style_adjusted:
        return raw_distance.astype(np.float32), available

    # Preserve absolute enthusiasm and compare the shape of priorities separately.
    means = np.nanmean(values, axis=1)
    centered = values - means[:, None]
    shape_sum = np.zeros((n, n), dtype=np.float32)
    shape_count = np.zeros((n, n), dtype=np.int16)
    for j in range(m):
        x = centered[:, j]
        valid = row_available & np.isfinite(x)
        available_j = pair_availability(valid)
        # A centered 1-5 battery item can differ by at most 8 across two respondents.
        d = np.abs(x[:, None] - x[None, :]) / (2 * scale_range)
        shape_sum += np.where(available_j, d, 0).astype(np.float32)
        shape_count += available_j.astype(np.int16)
    shape_distance = np.divide(shape_sum, shape_count, out=np.zeros_like(shape_sum), where=shape_count > 0)
    mean_distance, mean_available = pairwise_single_numeric(means, scale_range, row_available)
    combined_available = available & mean_available
    combined = 0.5 * raw_distance + 0.5 * shape_distance
    combined[~combined_available] = 0.0
    return combined.astype(np.float32), combined_available


def encode_unit(
    frame: pd.DataFrame,
    unit: DistanceUnit,
    prospect_fit_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    row_available = availability_vector(frame, unit)

    if unit.kind == 'ordinal':
        values = frame[unit.columns[0]].map(unit.mapping).to_numpy(dtype=float)
        return pairwise_single_numeric(values, float(unit.ordered_range), row_available)

    if unit.kind == 'nominal':
        values = frame[unit.columns[0]].astype(object).to_numpy()
        return pairwise_nominal(values, row_available)

    if unit.kind == 'single_choice_options':
        binary = frame[list(unit.columns)].notna().to_numpy(dtype=int)
        row_total = binary.sum(axis=1)
        assert np.all(row_total == 1), f'{unit.name}: expected one selected option per respondent.'
        values = binary.argmax(axis=1).astype(object)
        return pairwise_nominal(values, row_available)

    if unit.kind == 'binary_selected':
        values = frame[unit.columns[0]].notna().to_numpy(dtype=float)
        return pairwise_single_numeric(values, 1.0, row_available)

    if unit.kind == 'numeric':
        values = pd.to_numeric(frame[unit.columns[0]], errors='coerce').to_numpy(dtype=float)
        if unit.log1p:
            values = np.log1p(np.clip(values, 0, None))
        fit_values = values[prospect_fit_mask & np.isfinite(values)]
        cap = float(np.quantile(fit_values, unit.cap_quantile))
        values = np.minimum(values, cap)
        fit_values = values[prospect_fit_mask & np.isfinite(values)]
        value_range = float(fit_values.max() - fit_values.min())
        if value_range == 0:
            raise ValueError(f'{unit.name}: zero prospect-fitted numeric range.')
        return pairwise_single_numeric(values, value_range, row_available)

    if unit.kind == 'multiselect':
        binary = frame[list(unit.columns)].notna().to_numpy(dtype=int)
        return pairwise_multiselect(binary, row_available)

    if unit.kind == 'likert_battery':
        values = frame[list(unit.columns)].apply(lambda s: s.map(unit.mapping)).to_numpy(dtype=float)
        return pairwise_likert_battery(values, float(unit.ordered_range), row_available, response_style_adjusted=False)

    raise ValueError(f'Unknown unit kind: {unit.kind}')

# Unit tests for the critical Jaccard rule.
toy = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=int)
toy_d, _ = pairwise_multiselect(toy, np.ones(4, dtype=bool))
assert toy_d[0, 1] == 0.0       # identical selections
assert toy_d[0, 2] == 1.0       # disjoint selections
assert toy_d[0, 3] == 1.0       # shared zeroes do not create similarity
assert toy_d[3, 3] == 0.0       # two empty sets

@dataclass
class CachedUnitDistance:
    unit: DistanceUnit
    raw: np.ndarray
    adjusted: np.ndarray
    available: np.ndarray


def build_unit_distance_cache(
    frame: pd.DataFrame,
    units: Sequence[DistanceUnit],
    prospect_fit_mask: np.ndarray,
) -> tuple[dict[str, CachedUnitDistance], pd.DataFrame]:
    cache: dict[str, CachedUnitDistance] = {}
    qa_rows = []
    for unit in units:
        row_available = availability_vector(frame, unit)
        if unit.kind == 'likert_battery':
            values = frame[list(unit.columns)].apply(lambda s: s.map(unit.mapping)).to_numpy(dtype=float)
            raw_distance, available = pairwise_likert_battery(
                values, float(unit.ordered_range), row_available, response_style_adjusted=False
            )
            adjusted_distance, adjusted_available = pairwise_likert_battery(
                values, float(unit.ordered_range), row_available, response_style_adjusted=True
            )
            assert np.array_equal(available, adjusted_available)
        else:
            raw_distance, available = encode_unit(frame, unit, prospect_fit_mask)
            adjusted_distance = raw_distance
        cache[unit.name] = CachedUnitDistance(
            unit=unit,
            raw=raw_distance.astype(np.float32),
            adjusted=adjusted_distance.astype(np.float32),
            available=available.astype(bool),
        )
        qa_rows.append({
            'unit': unit.name,
            'block': unit.block,
            'kind': unit.kind,
            'mean_pair_distance_raw': float(raw_distance[available].mean()) if available.any() else np.nan,
            'mean_pair_distance_adjusted': float(adjusted_distance[available].mean()) if available.any() else np.nan,
            'available_pair_share': float(available.mean()),
        })
    return cache, pd.DataFrame(qa_rows)


def register_external_numeric_unit(
    cache: dict[str, CachedUnitDistance],
    name: str,
    block: str,
    values: pd.Series,
    prospect_fit_mask: np.ndarray,
    fixed_range: float | None = None,
) -> None:
    """Add a derived respondent-level score without changing raw survey-unit code."""
    array = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    valid = np.isfinite(array)
    fit_values = array[prospect_fit_mask & valid]
    if fixed_range is None:
        lower, upper = np.quantile(fit_values, [0.01, 0.99])
        array = np.clip(array, lower, upper)
        value_range = float(upper - lower)
    else:
        value_range = float(fixed_range)
    if not np.isfinite(value_range) or value_range <= 0:
        raise ValueError(f'{name}: invalid external numeric range={value_range}')
    distance, available = pairwise_single_numeric(array, value_range, np.ones(len(array), dtype=bool))
    unit = DistanceUnit(name=name, block=block, kind='external_numeric', columns=(name,))
    cache[name] = CachedUnitDistance(unit=unit, raw=distance, adjusted=distance, available=available)


def validate_segment_experiment(config: SegmentExperiment, cache: Mapping[str, CachedUnitDistance]) -> list[str]:
    cache_units = set(cache)
    cache_blocks = {entry.unit.block for entry in cache.values()}
    unknown_blocks = set(config.include_blocks).difference(cache_blocks)
    if unknown_blocks:
        raise KeyError(f'{config.name}: unknown blocks={sorted(unknown_blocks)}')
    if config.include_units is not None:
        unknown = set(config.include_units).difference(cache_units)
        if unknown:
            raise KeyError(f'{config.name}: unknown included units={sorted(unknown)}')
    unknown_excluded = set(config.exclude_units).difference(cache_units)
    if unknown_excluded:
        raise KeyError(f'{config.name}: unknown excluded units={sorted(unknown_excluded)}')
    unknown_weighted = set(config.unit_weights).difference(cache_units)
    if unknown_weighted:
        raise KeyError(f'{config.name}: unknown weighted units={sorted(unknown_weighted)}')
    unknown_weighted_blocks = set(config.block_weights).difference(config.include_blocks)
    if unknown_weighted_blocks:
        raise KeyError(f'{config.name}: weights supplied for excluded blocks={sorted(unknown_weighted_blocks)}')

    selected = [
        name for name, entry in cache.items()
        if entry.unit.block in config.include_blocks
        and (config.include_units is None or name in config.include_units)
        and name not in config.exclude_units
        and float(config.unit_weights.get(name, 1.0)) > 0
    ]
    if not selected:
        raise ValueError(f'{config.name}: no active units.')
    missing_blocks = set(config.include_blocks).difference({cache[name].unit.block for name in selected})
    if missing_blocks:
        raise ValueError(f'{config.name}: included blocks without active units={sorted(missing_blocks)}')
    if any(float(value) <= 0 for value in config.block_weights.values()):
        raise ValueError(f'{config.name}: block weights must be positive.')
    if any(float(value) < 0 for value in config.unit_weights.values()):
        raise ValueError(f'{config.name}: unit weights cannot be negative.')
    return selected


def assemble_experiment_distance(
    cache: Mapping[str, CachedUnitDistance],
    config: SegmentExperiment,
) -> tuple[np.ndarray, pd.DataFrame, pd.DataFrame]:
    selected = validate_segment_experiment(config, cache)
    shape = next(iter(cache.values())).raw.shape
    block_rows = []
    feature_rows = []
    block_distances: dict[str, np.ndarray] = {}
    block_available: dict[str, np.ndarray] = {}

    for block in config.include_blocks:
        names = [name for name in selected if cache[name].unit.block == block]
        numerator = np.zeros(shape, dtype=np.float32)
        denominator = np.zeros(shape, dtype=np.float32)
        for name in names:
            entry = cache[name]
            weight = float(config.unit_weights.get(name, 1.0))
            distance = entry.adjusted if config.response_style_adjusted else entry.raw
            numerator += weight * np.where(entry.available, distance, 0.0).astype(np.float32)
            denominator += weight * entry.available.astype(np.float32)
            feature_rows.append({
                'experiment': config.name,
                'unit': name,
                'block': block,
                'unit_weight': weight,
                'response_style_adjusted': config.response_style_adjusted,
            })
        available = denominator > 0
        block_distance = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=available)
        block_distances[block] = block_distance.astype(np.float32)
        block_available[block] = available
        block_rows.append({
            'experiment': config.name,
            'block': block,
            'block_weight': float(config.block_weights.get(block, 1.0)),
            'active_unit_n': len(names),
            'available_pair_share': float(available.mean()),
        })

    total_numerator = np.zeros(shape, dtype=np.float32)
    total_denominator = np.zeros(shape, dtype=np.float32)
    for block in config.include_blocks:
        weight = float(config.block_weights.get(block, 1.0))
        available = block_available[block]
        total_numerator += weight * np.where(available, block_distances[block], 0.0).astype(np.float32)
        total_denominator += weight * available.astype(np.float32)
    total_available = total_denominator > 0
    if not total_available.all():
        raise ValueError(f'{config.name}: {int((~total_available).sum())} respondent pairs have no comparable features.')
    total = np.divide(total_numerator, total_denominator, out=np.zeros_like(total_numerator), where=total_available)
    total = (total + total.T) / 2
    np.fill_diagonal(total, 0.0)
    assert np.nanmin(total) >= -1e-7 and np.nanmax(total) <= 1 + 1e-7
    return total.astype(np.float32), pd.DataFrame(feature_rows), pd.DataFrame(block_rows)


In [7]:
@dataclass
class KMedoidsResult:
    labels: np.ndarray
    medoid_indices: np.ndarray
    cost: float
    n_iter: int


def init_kmedoids_pp(D: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    n = D.shape[0]
    medoids = [int(rng.integers(n))]
    min_distance = D[:, medoids[0]].astype(float).copy()
    while len(medoids) < k:
        probabilities = min_distance ** 2
        probabilities[medoids] = 0.0
        if probabilities.sum() == 0:
            candidates = np.setdiff1d(np.arange(n), np.asarray(medoids), assume_unique=False)
            next_medoid = int(rng.choice(candidates))
        else:
            probabilities /= probabilities.sum()
            next_medoid = int(rng.choice(n, p=probabilities))
        medoids.append(next_medoid)
        min_distance = np.minimum(min_distance, D[:, next_medoid])
    return np.asarray(medoids, dtype=int)


def fit_kmedoids(
    D: np.ndarray,
    k: int,
    seed: int = SEED,
    n_init: int = KMEDOIDS_N_INIT,
    max_iter: int = KMEDOIDS_MAX_ITER,
) -> KMedoidsResult:
    if D.shape[0] != D.shape[1]:
        raise ValueError('D must be square.')
    best: KMedoidsResult | None = None
    for init_id in range(n_init):
        rng = np.random.default_rng(seed + 1009 * init_id + 7919 * k)
        medoids = init_kmedoids_pp(D, k, rng)
        n_iter_done = 0
        for iteration in range(max_iter):
            distances_to_medoids = D[:, medoids]
            labels = distances_to_medoids.argmin(axis=1)
            new_medoids = medoids.copy()
            for cluster in range(k):
                members = np.flatnonzero(labels == cluster)
                if len(members) == 0:
                    nearest = distances_to_medoids.min(axis=1)
                    candidates = np.setdiff1d(np.argsort(nearest)[::-1], new_medoids, assume_unique=False)
                    new_medoids[cluster] = int(candidates[0])
                    continue
                within = D[np.ix_(members, members)]
                new_medoids[cluster] = int(members[np.argmin(within.sum(axis=1))])
            n_iter_done = iteration + 1
            if np.array_equal(np.sort(new_medoids), np.sort(medoids)):
                medoids = new_medoids
                break
            medoids = new_medoids
        final_distances = D[:, medoids]
        labels = final_distances.argmin(axis=1)
        cost = float(final_distances[np.arange(len(D)), labels].sum())
        result = KMedoidsResult(labels=labels, medoid_indices=medoids, cost=cost, n_iter=n_iter_done)
        if best is None or result.cost < best.cost:
            best = result
    assert best is not None
    assert len(np.unique(best.medoid_indices)) == k
    assert len(np.unique(best.labels)) == k
    return best

def matched_cluster_jaccards(reference: np.ndarray, candidate: np.ndarray, k: int) -> np.ndarray:
    overlap = np.zeros((k, k), dtype=int)
    for i in range(k):
        for j in range(k):
            overlap[i, j] = np.sum((reference == i) & (candidate == j))
    row_ind, col_ind = linear_sum_assignment(-overlap)
    result = np.zeros(k, dtype=float)
    for ref_cluster, cand_cluster in zip(row_ind, col_ind):
        ref_set = reference == ref_cluster
        cand_set = candidate == cand_cluster
        union = np.sum(ref_set | cand_set)
        result[ref_cluster] = np.sum(ref_set & cand_set) / union if union else np.nan
    return result


def stability_for_solution(
    D: np.ndarray,
    reference_labels: np.ndarray,
    k: int,
    runs: int,
    fraction: float = STABILITY_FRACTION,
    seed: int = SEED,
    n_init: int = 3,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed + 313 * k)
    sample_n = int(round(len(D) * fraction))
    run_rows = []
    cluster_rows = []
    for run_id in range(runs):
        sample = np.sort(rng.choice(len(D), size=sample_n, replace=False))
        result = fit_kmedoids(D[np.ix_(sample, sample)], k=k, seed=seed + run_id, n_init=n_init)
        reference_sub = reference_labels[sample]
        ari = adjusted_rand_score(reference_sub, result.labels)
        jaccards = matched_cluster_jaccards(reference_sub, result.labels, k)
        run_rows.append({
            'run': run_id,
            'ARI': ari,
            'mean_cluster_jaccard': np.nanmean(jaccards),
            'min_cluster_jaccard': np.nanmin(jaccards),
        })
        for cluster, value in enumerate(jaccards):
            cluster_rows.append({'run': run_id, 'reference_cluster': cluster, 'jaccard': value})
    return pd.DataFrame(run_rows), pd.DataFrame(cluster_rows)


@dataclass
class SegmentExperimentRun:
    config: SegmentExperiment
    D_all: np.ndarray
    D_prospects: np.ndarray
    active_features: pd.DataFrame
    active_blocks: pd.DataFrame
    candidate_results: dict[int, KMedoidsResult]
    candidate_summary: pd.DataFrame
    stability_details: dict[int, tuple[pd.DataFrame, pd.DataFrame]]
    assignments: dict[int, pd.DataFrame]
    natural_fit: dict[int, pd.DataFrame]

def candidate_assignment_table(D_all: np.ndarray, result: KMedoidsResult) -> tuple[pd.DataFrame, pd.DataFrame]:
    medoid_all_positions = prospect_positions[result.medoid_indices]
    distance_to_medoids = D_all[:, medoid_all_positions]
    labels = distance_to_medoids.argmin(axis=1)
    sorted_distances = np.sort(distance_to_medoids, axis=1)
    table = retained_model[[ID_COL, SOURCE_COL, HISTORY_COL, LEGACY_CLUSTER_COL, '_raw_index']].copy()
    table['persona_cluster'] = labels
    table['nearest_medoid_distance'] = sorted_distances[:, 0]
    table['assignment_margin'] = sorted_distances[:, 1] - sorted_distances[:, 0]
    table['is_prospect_fit_population'] = prospect_fit_mask
    table['is_current_past_donor_cohort'] = donor_mask
    assert np.array_equal(table.loc[prospect_fit_mask, 'persona_cluster'].to_numpy(), result.labels)

    fit = pd.crosstab(table['persona_cluster'], table[SOURCE_COL])
    for column in ['Panel', 'List']:
        if column not in fit:
            fit[column] = 0
    fit = fit[['Panel', 'List']].rename(columns={'Panel': 'prospect_n', 'List': 'past_donor_n'})
    fit['prospect_share'] = fit['prospect_n'] / fit['prospect_n'].sum()
    fit['past_donor_share'] = fit['past_donor_n'] / fit['past_donor_n'].sum()
    fit['natural_fit_index'] = fit['past_donor_share'] / fit['prospect_share']
    fit['past_donor_share_of_persona'] = fit['past_donor_n'] / fit[['prospect_n', 'past_donor_n']].sum(axis=1)
    confidence = table.groupby('persona_cluster')['assignment_margin'].agg(['median', 'mean']).rename(
        columns={'median': 'median_assignment_margin', 'mean': 'mean_assignment_margin'}
    )
    return table, fit.join(confidence)


def candidate_guardrail_metrics(
    labels: np.ndarray,
    active_unit_names: Sequence[str],
    config: SegmentExperiment,
) -> dict[str, Any]:
    prospect_demo = segment_guardrails.loc[prospect_fit_mask].copy()
    cluster_demo = prospect_demo.groupby(pd.Series(labels, index=prospect_demo.index)).mean()
    demo_gap = cluster_demo.sub(prospect_demo.mean(), axis=1).abs()
    max_demo_position = np.unravel_index(np.nanargmax(demo_gap.to_numpy()), demo_gap.shape)
    max_demo_feature = demo_gap.columns[max_demo_position[1]]
    max_demo_gap = float(demo_gap.iloc[max_demo_position])

    intensity = pd.DataFrame({'cluster': labels, 'value': retained_response_intensity[prospect_fit_mask]})
    intensity_range = float(intensity.groupby('cluster')['value'].mean().max() - intensity.groupby('cluster')['value'].mean().min())

    unit_scores = {}
    for name in active_unit_names:
        entry = UNIT_DISTANCE_CACHE[name]
        unit_distance = entry.adjusted if config.response_style_adjusted else entry.raw
        unit_prospects = unit_distance[np.ix_(prospect_fit_mask, prospect_fit_mask)]
        try:
            unit_scores[name] = float(silhouette_score(
                unit_prospects,
                labels,
                metric='precomputed',
                sample_size=min(300, len(labels)),
                random_state=SEED,
            ))
        except Exception:
            unit_scores[name] = np.nan
    dominant_unit = max(unit_scores, key=lambda key: -np.inf if np.isnan(unit_scores[key]) else unit_scores[key])

    writein_counts = pd.DataFrame({
        'cluster': labels,
        'value': charity_model_features.loc[prospect_fit_mask, 'writein_valid_mention_count'].to_numpy(dtype=float),
    })
    writein_count_range = float(
        writein_counts.groupby('cluster')['value'].mean().max()
        - writein_counts.groupby('cluster')['value'].mean().min()
    )
    if charity_common_entity_presence_model.shape[1]:
        entity_rates = charity_common_entity_presence_model.groupby(
            pd.Series(labels, index=charity_common_entity_presence_model.index)
        ).mean()
        overall_entity_rates = charity_common_entity_presence_model.mean()
        max_common_entity_gap = float(entity_rates.sub(overall_entity_rates, axis=1).abs().to_numpy().max())
    else:
        max_common_entity_gap = np.nan

    return {
        'response_intensity_range': intensity_range,
        'max_demographic_pp_gap': max_demo_gap,
        'max_demographic_feature': max_demo_feature,
        'dominant_unit': dominant_unit,
        'max_single_unit_silhouette': unit_scores[dominant_unit],
        'writein_mention_count_range': writein_count_range,
        'max_common_entity_pp_gap': max_common_entity_gap,
    }


def run_segment_experiment(
    config: SegmentExperiment,
    cache: Mapping[str, CachedUnitDistance],
    mode: str,
) -> SegmentExperimentRun:
    settings = EXPERIMENT_MODES[mode]
    D_all, active_features, active_blocks = assemble_experiment_distance(cache, config)
    D_prospects = D_all[np.ix_(prospect_fit_mask, prospect_fit_mask)]
    active_names = active_features['unit'].tolist()
    results = {}
    stability = {}
    assignments = {}
    natural_fit = {}
    rows = []

    for k in config.k_values:
        result = fit_kmedoids(D_prospects, k=k, n_init=settings['fit_n_init'])
        run_stability, cluster_stability = stability_for_solution(
            D_prospects,
            result.labels,
            k=k,
            runs=settings['stability_runs'],
            fraction=STABILITY_FRACTION,
            n_init=settings['stability_n_init'],
        )
        assignment, donor_fit = candidate_assignment_table(D_all, result)
        guardrails = candidate_guardrail_metrics(result.labels, active_names, config)
        cluster_sizes = pd.Series(result.labels).value_counts()
        top_fit_cluster = int(donor_fit['natural_fit_index'].idxmax())
        rows.append({
            'experiment': config.name,
            'k': k,
            'silhouette': silhouette_score(D_prospects, result.labels, metric='precomputed'),
            'mean_distance_to_medoid': result.cost / len(result.labels),
            'minimum_cluster_n': int(cluster_sizes.min()),
            'minimum_cluster_share': float(cluster_sizes.min() / len(result.labels)),
            'mean_stability_ARI': float(run_stability['ARI'].mean()),
            'weakest_cluster_median_jaccard': float(cluster_stability.groupby('reference_cluster')['jaccard'].median().min()),
            'top_natural_fit_cluster': top_fit_cluster,
            'max_natural_fit_index': float(donor_fit.loc[top_fit_cluster, 'natural_fit_index']),
            'max_past_donor_share': float(donor_fit['past_donor_share'].max()),
            'median_assignment_margin': float(assignment.loc[prospect_fit_mask, 'assignment_margin'].median()),
            **guardrails,
        })
        results[k] = result
        stability[k] = (run_stability, cluster_stability)
        assignments[k] = assignment
        natural_fit[k] = donor_fit

    summary = pd.DataFrame(rows).set_index('k')
    run = SegmentExperimentRun(
        config=config,
        D_all=D_all,
        D_prospects=D_prospects,
        active_features=active_features,
        active_blocks=active_blocks,
        candidate_results=results,
        candidate_summary=summary,
        stability_details=stability,
        assignments=assignments,
        natural_fit=natural_fit,
    )

    experiment_dir = OUTPUT_DIR / 'segment_experiments' / config.name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    config_payload = config.as_dict() | {
        'engine_version': SEGMENT_ENGINE_VERSION,
        'mode': mode,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'input_sha256': {row['artifact']: row['sha256'] for _, row in input_manifest.iterrows()},
    }
    (experiment_dir / 'config.json').write_text(json.dumps(config_payload, indent=2, sort_keys=True))
    active_features.to_csv(experiment_dir / 'active_features.csv', index=False)
    active_blocks.to_csv(experiment_dir / 'active_blocks.csv', index=False)
    summary.to_csv(experiment_dir / 'candidate_summary.csv')
    for k in config.k_values:
        assignments[k].assign(experiment=config.name, k=k).to_csv(experiment_dir / f'assignments_k{k}.csv', index=False)
        medoid_rows = retained_model.iloc[prospect_positions[results[k].medoid_indices]][[ID_COL, '_raw_index']].copy()
        medoid_rows.insert(0, 'cluster', np.arange(k))
        medoid_rows.to_csv(experiment_dir / f'medoids_k{k}.csv', index=False)
        stability[k][0].to_csv(experiment_dir / f'stability_runs_k{k}.csv', index=False)
        stability[k][1].to_csv(experiment_dir / f'cluster_stability_k{k}.csv', index=False)
    return run


In [8]:
@dataclass(frozen=True)
class ProfileSpec:
    name: str
    label: str
    group: str
    kind: str
    values: pd.Series


def slug(value: str, limit: int = 55) -> str:
    return re.sub(r'[^a-z0-9]+', '_', value.casefold()).strip('_')[:limit]


def question_options(question: str, *, exclude_text: bool = True) -> list[str]:
    prefix = f'{question} - '
    labels = []
    for column in retained_model.columns:
        if not column.startswith(prefix):
            continue
        label = column[len(prefix):]
        if exclude_text and ('Text_' in label or label.endswith('Other- Text')):
            continue
        labels.append(label)
    return labels


def build_profile_registry(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    specs: list[ProfileSpec] = []

    def add(name: str, label: str, group: str, values: pd.Series, kind: str = 'binary') -> None:
        specs.append(ProfileSpec(name, label, group, kind, pd.Series(values.to_numpy(), index=frame.index)))

    def category(column: str, group: str, prefix: str) -> None:
        for value in sorted(frame[column].dropna().astype(str).unique()):
            add(f'{prefix}__{slug(value, 45)}', value, group, frame[column].eq(value))

    def multiselect(question: str, group: str, prefix: str, options: Sequence[str] | None = None) -> None:
        for option in options or question_options(question):
            try:
                values = selected(frame, question, option)
            except KeyError:
                continue
            add(f'{prefix}__{slug(option)}', option, group, values)

    age = pd.to_numeric(frame[AGE_COL], errors='coerce')
    add('age_years', 'Average age', 'Demographics', age, 'numeric')
    for name, label, mask in (
        ('age_18_34', 'Age 18-34', age.between(18, 34)),
        ('age_35_54', 'Age 35-54', age.between(35, 54)),
        ('age_55_plus', 'Age 55+', age.ge(55)),
    ):
        add(name, label, 'Demographics', mask)

    for column, group, prefix in (
        (GENDER_COL, 'Demographics', 'gender'),
        (INCOME_COL, 'Demographics', 'income'),
        (EDUCATION_COL, 'Demographics', 'education'),
        (EMPLOYMENT_COL, 'Demographics', 'employment'),
        ('What is your marital status?', 'Demographics', 'marital'),
        ('Including yourself, how many people are currently living in your household?', 'Demographics', 'household'),
        (REGION_COL, 'Geography', 'region'),
        (URBANICITY_COL, 'Geography', 'urbanicity'),
        (FREQUENCY_COL, 'Giving behavior', 'frequency'),
        (AMOUNT_COL, 'Giving behavior', 'amount'),
        (DECISION_COL, 'Giving behavior', 'decision'),
    ):
        category(column, group, prefix)

    for question, group, prefix in (
        (RACE_QUESTION, 'Demographics', 'race'),
        (CHILDREN_QUESTION, 'Demographics', 'children'),
        (Q_CURRENT_CAUSES, 'Current causes', 'cause'),
        (Q_EDU_FORMATS, 'Education formats', 'education_format'),
        (Q_MOTIVATIONS, 'Motivations', 'motivation'),
        (Q_DISCOVERY, 'Discovery', 'discovery'),
        (Q_RESEARCH_SOURCES, 'Research channels', 'research'),
        (Q_AD_RESPONSE, 'Advertising response', 'ad_response'),
        (Q_MEDIA, 'Media', 'media'),
        (Q_SOCIAL_PERSONAL, 'Social media', 'social_personal'),
        (Q_SOCIAL_NEWS, 'Social media', 'social_news'),
        (Q_SCHOOL_INVOLVEMENT, 'School involvement', 'school'),
    ):
        multiselect(question, group, prefix)

    add('organizations_per_year', 'Average organizations supported per year', 'Giving behavior', pd.to_numeric(frame[ORG_COUNT_COL], errors='coerce'), 'numeric')
    add('donations_per_year', 'Average donations per year', 'Giving behavior', pd.to_numeric(frame[DONATION_COUNT_COL], errors='coerce'), 'numeric')

    for item, label, group, question, mapping in (
        ('I research an organization thoroughly before making a donation', 'Researches organizations thoroughly', 'Decision style', Q_CHARITY_ATTITUDES, AGREE_MAP),
        ('I tend to stick to organizations that I trust and prefer and do not seek to find new ones for potential donations', 'Sticks to familiar organizations', 'Decision style', Q_CHARITY_ATTITUDES, AGREE_MAP),
        ('I am more likely to donate during times of crisis or disaster', 'Crisis-oriented', 'Activation', Q_CHARITY_ATTITUDES, AGREE_MAP),
        ('Understanding the specific impact of your donation (e.g., how funds are used, success stories)', 'Specific-impact message', 'Messaging', Q_MESSAGING, INFLUENTIAL_MAP),
        ("Information about the charity's transparency and financial accountability", 'Transparency message', 'Messaging', Q_MESSAGING, INFLUENTIAL_MAP),
    ):
        top_values = set(list(mapping)[-2:])
        add(slug(label), label, group, frame[likert_col(item, question)].isin(top_values))

    values = pd.DataFrame({spec.name: spec.values for spec in specs}, index=frame.index)
    metadata = pd.DataFrame([
        {'feature': spec.name, 'label': spec.label, 'group': spec.group, 'kind': spec.kind}
        for spec in specs
    ]).drop_duplicates('feature').set_index('feature')
    assert metadata.index.equals(pd.Index(values.columns))
    return values, metadata


def profile_solution(
    run: SegmentExperimentRun,
    k: int,
    profile_values: pd.DataFrame,
    profile_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    assignments = run.assignments[k]
    labels = assignments.loc[prospect_fit_mask, 'persona_cluster'].to_numpy()
    values = profile_values.loc[prospect_fit_mask]
    overall = values.mean(numeric_only=True)
    overall_sd = values.std(numeric_only=True).replace(0, np.nan)
    cluster_means = values.groupby(pd.Series(labels, index=values.index, name='cluster')).mean(numeric_only=True)

    rows = []
    for cluster, cluster_row in cluster_means.iterrows():
        for feature, cluster_value in cluster_row.items():
            meta = profile_metadata.loc[feature]
            overall_value = float(overall[feature])
            difference = float(cluster_value - overall_value)
            lift = float(cluster_value / overall_value) if meta['kind'] == 'binary' and overall_value > 0 else np.nan
            standardized = difference / overall_sd[feature] if meta['kind'] != 'binary' and pd.notna(overall_sd[feature]) else np.nan
            magnitude = abs(lift - 1) if meta['kind'] == 'binary' and pd.notna(lift) else abs(standardized) if pd.notna(standardized) else abs(difference)
            rows.append({
                'cluster': int(cluster), 'feature': feature, 'label': meta['label'],
                'group': meta['group'], 'kind': meta['kind'],
                'cluster_value': float(cluster_value), 'overall_value': overall_value,
                'difference': difference, 'lift': lift,
                'standardized_difference': standardized,
                'diagnostic_magnitude': float(magnitude),
            })

    tidy = pd.DataFrame(rows)
    sizes = assignments.loc[prospect_fit_mask, 'persona_cluster'].value_counts().sort_index().rename('prospect_n').to_frame()
    sizes['prospect_share'] = sizes['prospect_n'] / sizes['prospect_n'].sum()
    natural_fit = run.natural_fit[k].copy()
    summary = sizes.join(natural_fit.drop(columns=sizes.columns.intersection(natural_fit.columns)), how='left')
    return tidy, summary


def rectangular_cluster_match(left: np.ndarray, right: np.ndarray) -> tuple[list[tuple[int, int]], float]:
    left_values, right_values = np.unique(left), np.unique(right)
    overlap = np.array([[(left == a).astype(int) @ (right == b).astype(int) for b in right_values] for a in left_values])
    rows, cols = linear_sum_assignment(-overlap)
    pairs = [(int(left_values[i]), int(right_values[j])) for i, j in zip(rows, cols)]
    return pairs, float(overlap[rows, cols].sum() / len(left))


def top_profile_overlap(left: pd.DataFrame, right: pd.DataFrame, pairs: Sequence[tuple[int, int]], n: int = 10) -> float:
    excluded = {'Demographics', 'Geography', 'Media', 'Social media'}
    overlaps = []
    for left_cluster, right_cluster in pairs:
        left_features = set(left[(left.cluster == left_cluster) & ~left.group.isin(excluded)].nlargest(n, 'diagnostic_magnitude').feature)
        right_features = set(right[(right.cluster == right_cluster) & ~right.group.isin(excluded)].nlargest(n, 'diagnostic_magnitude').feature)
        union = left_features | right_features
        overlaps.append(len(left_features & right_features) / len(union) if union else np.nan)
    return float(np.nanmean(overlaps))


In [9]:
JTBD_SEED = SEED + 7000
JTBD_STABILITY_FRACTION = 0.80
JTBD_FACTOR_MIN = 3
JTBD_FACTOR_MAX = 8
jtbd_retained = retained.copy(deep=True)
jtbd_prospects = jtbd_retained.loc[prospect_mask_retained].copy(deep=True)
JTBD_JOBS = {
    'visible_impact': 'Produce a concrete, understandable result',
    'donor_control': 'Choose the recipient, scope, or use of the contribution',
    'relational_support': 'Show up for a person or institution I care about',
    'local_stewardship': 'Strengthen the community where I live',
    'urgent_action': 'Act when a need becomes urgent and emotionally real',
    'values_obligation': 'Put values, faith, identity, or duty into action',
    'giving_leverage': 'Make limited resources accomplish more',
    'sustained_giving': 'Make generosity a dependable practice',
    'confidence_risk': 'Avoid an unsafe, ineffective, or regrettable giving decision',
    'impact_witness': 'Follow the work and receive evidence of progress',
    'active_participation': 'Take part in the work beyond giving money',
}

Q_PRIOR_ACTIONS = 'Which of the following have you done in the past 12 months? Select all that apply.'
Q_ACTIVITIES = 'Please indicate the interests and activities in which you participate on a regular basis. Select all that apply.'
Q_SCHOOL = Q_SCHOOL_INVOLVEMENT
Q_CHILDREN_HOME = CHILDREN_QUESTION


def jtbd_cfg(
    feature_id: str,
    job_family: str | None,
    role: str,
    response_type: str,
    question_group: str,
    source_question: str,
    item: str | None = None,
    *,
    factor_pool: bool = False,
    adjustment: str = 'none',
    mapping_name: str | None = None,
    notes: str = '',
) -> dict[str, Any]:
    if response_type == 'likert':
        source_column = likert_col(item, source_question)
    elif response_type == 'binary':
        source_column = option_col(source_question, item)
    elif response_type in {'single_choice', 'frequency', 'derived'}:
        source_column = source_question
    else:
        raise ValueError(f'Unsupported response_type={response_type!r}')
    return {
        'feature_id': feature_id,
        'job_family': job_family,
        'role': role,
        'response_type': response_type,
        'question_group': question_group,
        'source_question': source_question,
        'item': item,
        'source_column': source_column,
        'factor_pool': bool(factor_pool),
        'adjustment': adjustment,
        'mapping_name': mapping_name,
        'score_direction': 1,
        'notes': notes,
    }


rows: list[dict[str, Any]] = []
add = rows.append

# 1. Visible impact
add(jtbd_cfg('visible_specific_impact', 'visible_impact', 'core_score', 'likert', 'Q8.1', Q_MESSAGING,
             'Understanding the specific impact of your donation (e.g., how funds are used, success stories)',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
add(jtbd_cfg('visible_material_over_awareness', 'visible_impact', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I believe material impact is more important than advocacy or awareness-building',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('visible_belief_in_impact', 'visible_impact', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I believe charitable donations make a significant impact on the community',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('visible_effectiveness_criterion', 'visible_impact', 'hiring_criterion', 'likert', 'Q3.1', Q_IMPORTANCE,
             "Effectiveness and impact of the organization's programs", adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
add(jtbd_cfg('visible_efficiency_criterion', 'visible_impact', 'hiring_criterion', 'likert', 'Q3.1', Q_IMPORTANCE,
             'Efficiency in using donations (low administrative costs)', adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
add(jtbd_cfg('visible_effectiveness_struggle', 'visible_impact', 'struggle', 'binary', 'Q7.2', Q_CHALLENGES,
             'Uncertainty about the effectiveness of my donation in making a real difference', adjustment='within_binary_question'))

# 2. Donor control and agency. Use one nonredundant Q7.1 recipient indicator.
add(jtbd_cfg('control_specific_projects', 'donor_control', 'core_score', 'binary', 'Q6.2', Q_RESEARCH_INFO,
             'Specific projects or initiatives currently underway', factor_pool=True, adjustment='within_binary_question',
             notes='A skipped research block is encoded as no observed selection in the absolute score and neutral after selection-style centering.'))
add(jtbd_cfg('control_communities_served', 'donor_control', 'core_score', 'binary', 'Q6.2', Q_RESEARCH_INFO,
             'Communities the organization serves', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('control_recipient_preselected', 'donor_control', 'core_score', 'derived', 'Q7.1', DECISION_COL,
             factor_pool=True, adjustment='raw_anchor',
             notes='1 when the cause/organization is selected before giving; amount certainty is retained only as a validator to avoid nested indicators.'))
add(jtbd_cfg('control_amount_preselected', 'donor_control', 'validator', 'derived', 'Q7.1', DECISION_COL,
             adjustment='raw_anchor'))
add(jtbd_cfg('control_use_struggle', 'donor_control', 'struggle', 'binary', 'Q7.2', Q_CHALLENGES,
             'Concerns about how my donation will be used', adjustment='within_binary_question'))
add(jtbd_cfg('control_gofundme_validator', 'donor_control', 'validator', 'binary', 'Q4.3', Q_CURRENT_CAUSES,
             'GoFundMe fundraisers', adjustment='within_binary_question'))

# 3. Relational support
add(jtbd_cfg('relational_importance', 'relational_support', 'core_score', 'likert', 'Q3.1', Q_IMPORTANCE,
             'Personal connection to the organization/cause', factor_pool=True, adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
for feature_id, item in [
    ('relational_motivation', 'Personal connection to the cause or issue'),
    ('relational_knows_involved', 'I know someone directly involved with the charity'),
    ('relational_friend_family_affected', 'The charity/organization directly impacts a friend or family member'),
    ('relational_direct_ask', 'Someone from the charity asks for a donation'),
]:
    add(jtbd_cfg(feature_id, 'relational_support', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS, item,
                 factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('relational_personally_affected', 'relational_support', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I am more likely to donate to a cause if I know someone personally affected by it',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('relational_family_friend_discovery', 'relational_support', 'activation', 'binary', 'Q5.1', Q_DISCOVERY,
             'Recommendations from family, friends, or colleagues', adjustment='within_binary_question'))
add(jtbd_cfg('relational_community_request', 'relational_support', 'activation', 'binary', 'Q5.1', Q_DISCOVERY,
             'Direct requests from community figures (e.g., teachers, organizers)', adjustment='within_binary_question'))

# 4. Local community stewardship
add(jtbd_cfg('local_community_motivation', 'local_stewardship', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS,
             'Desire to make a positive impact in the community', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('local_scale_selected', 'local_stewardship', 'core_score', 'binary', 'Q4.2', Q_IMPACT_SCALE,
             'My local community', factor_pool=True, adjustment='within_binary_question',
             notes='Instrument says single choice; export is audited and observed selections are preserved.'))
add(jtbd_cfg('local_message', 'local_stewardship', 'core_score', 'likert', 'Q8.1', Q_MESSAGING,
             'Appeals that include a sense of community involvement or local impact',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
add(jtbd_cfg('local_charity_preference', 'local_stewardship', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I prefer to donate to local charities rather than national or international ones',
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('local_community_cause', 'local_stewardship', 'validator', 'binary', 'Q4.3', Q_CURRENT_CAUSES,
             'Community development/ local neighborhood initiatives', adjustment='within_binary_question'))

# 5. Urgent emotional action
for feature_id, item in [
    ('urgent_emotional_event', 'Emotional response to a specific event or campaign'),
    ('urgent_need_information', 'Information about the urgent need for support'),
]:
    add(jtbd_cfg(feature_id, 'urgent_action', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS, item,
                 factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('urgent_message', 'urgent_action', 'core_score', 'likert', 'Q8.1', Q_MESSAGING,
             'Messages that emphasize the urgency of the situation or need', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
add(jtbd_cfg('urgent_crisis_attitude', 'urgent_action', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I am more likely to donate during times of crisis or disaster', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
for feature_id, question, item in [
    ('urgent_story_discovery', Q_DISCOVERY, 'Stories or testimonials'),
    ('urgent_social_discovery', Q_DISCOVERY, 'Social media campaigns or influencers promoting a cause'),
    ('urgent_celebrity_discovery', Q_DISCOVERY, 'Celebrity endorsements in a charity'),
    ('urgent_social_ad', Q_AD_RESPONSE, 'Ads embedded in social media apps'),
]:
    add(jtbd_cfg(feature_id, 'urgent_action', 'activation', 'binary', 'activation', question, item,
                 adjustment='within_binary_question'))
for feature_id, item in [
    ('urgent_disaster_validator', 'Disaster relief and humanitarian aid organizations'),
    ('urgent_housing_validator', 'Homelessness and housing support services'),
]:
    add(jtbd_cfg(feature_id, 'urgent_action', 'validator', 'binary', 'Q4.3', Q_CURRENT_CAUSES, item,
                 adjustment='within_binary_question'))

# 6. Values and obligation
for feature_id, item in [
    ('values_mission_motivation', 'Belief in the mission and values of the organization'),
    ('values_religious_motivation', 'Religious or spiritual beliefs'),
]:
    add(jtbd_cfg(feature_id, 'values_obligation', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS, item,
                 factor_pool=True, adjustment='within_binary_question'))
for feature_id, question, item, mapping in [
    ('values_mission_message', Q_MESSAGING, 'A clear and compelling mission statement or vision from the charity', 'INFLUENTIAL_MAP'),
    ('values_alignment_message', Q_MESSAGING, 'Messages that align with your personal values and belief', 'INFLUENTIAL_MAP'),
    ('values_alignment_attitude', Q_CHARITY_ATTITUDES, 'I make donations to organizations/causes that align with my personal values', 'AGREE_MAP'),
    ('values_obligation_attitude', Q_CHARITY_ATTITUDES, 'I donate to charity out of a moral/religious obligation', 'AGREE_MAP'),
]:
    add(jtbd_cfg(feature_id, 'values_obligation', 'core_score', 'likert',
                 'Q8.1' if question == Q_MESSAGING else 'Q9.1', question, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name=mapping))
add(jtbd_cfg('values_diversity_criterion', 'values_obligation', 'hiring_criterion', 'likert', 'Q3.1', Q_IMPORTANCE,
             "Diversity and inclusivity in the organization's practices and leadership", adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
add(jtbd_cfg('values_religious_leader', 'values_obligation', 'activation', 'binary', 'Q5.1', Q_DISCOVERY,
             'Religious or spiritual leaders', adjustment='within_binary_question'))

# 7. Giving leverage
for feature_id, item in [
    ('leverage_tax_importance', 'Availability of tax deductions for donations'),
    ('leverage_match_importance', 'Presence of matching gift opportunities'),
    ('leverage_employer_match_importance', 'My employer matching the donation'),
]:
    add(jtbd_cfg(feature_id, 'giving_leverage', 'core_score', 'likert', 'Q3.1', Q_IMPORTANCE, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
for feature_id, item in [
    ('leverage_tax_motivation', 'Tax benefits or financial incentives'),
    ('leverage_employer_match_motivation', 'Corporate matching programs through my employer'),
]:
    add(jtbd_cfg(feature_id, 'giving_leverage', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS, item,
                 factor_pool=True, adjustment='within_binary_question'))
for feature_id, item in [
    ('leverage_match_message', 'Information about matching donation opportunities (e.g., employer match, challenge grants)'),
    ('leverage_tax_message', 'Information highlighting the tax benefits of donating'),
]:
    add(jtbd_cfg(feature_id, 'giving_leverage', 'core_score', 'likert', 'Q8.1', Q_MESSAGING, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
for feature_id, item in [
    ('leverage_tax_attitude', 'I am motivated to donate by tax benefits'),
    ('leverage_employer_match_attitude', 'I am more likely to donate to charities that my employer matches contributions to'),
]:
    add(jtbd_cfg(feature_id, 'giving_leverage', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))

# 8. Sustained giving practice
add(jtbd_cfg('sustained_habit_motivation', 'sustained_giving', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS,
             'Regular habit or tradition of giving', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('sustained_budget_attitude', 'sustained_giving', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I regularly set aside a portion of my income for charitable donations', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('sustained_frequency', 'sustained_giving', 'core_score', 'frequency', 'Q2.6', FREQUENCY_COL,
             factor_pool=True, adjustment='raw_anchor', mapping_name='FREQUENCY_MAP'))

# 9. Confidence and risk reduction
for feature_id, item in [
    ('confidence_transparency_importance', 'Transparency and accountability of the organization'),
    ('confidence_trusted_sources_importance', 'Recommendations or endorsements from trusted sources'),
    ('confidence_security_importance', 'Security and ease of the donation process'),
]:
    add(jtbd_cfg(feature_id, 'confidence_risk', 'core_score', 'likert', 'Q3.1', Q_IMPORTANCE, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
add(jtbd_cfg('confidence_trust_motivation', 'confidence_risk', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS,
             'Trust in the effectiveness and transparency of the organization', factor_pool=True,
             adjustment='within_binary_question'))
for feature_id, item in [
    ('confidence_transparency_message', "Information about the charity's transparency and financial accountability"),
    ('confidence_trusted_endorsement_message', 'Recommendations or endorsements from trusted sources (e.g., friends, family)'),
    ('confidence_ease_message', 'Messages that highlight the ease and convenience of the donation process'),
]:
    add(jtbd_cfg(feature_id, 'confidence_risk', 'core_score', 'likert', 'Q8.1', Q_MESSAGING, item,
                 factor_pool=True, adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
add(jtbd_cfg('confidence_research_attitude', 'confidence_risk', 'core_score', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I research an organization thoroughly before making a donation', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
add(jtbd_cfg('confidence_familiarity_route', 'confidence_risk', 'validator', 'likert', 'Q9.1', Q_CHARITY_ATTITUDES,
             'I tend to stick to organizations that I trust and prefer and do not seek to find new ones for potential donations',
             adjustment='within_ordered_battery', mapping_name='AGREE_MAP'))
for feature_id, item in [
    ('confidence_legitimacy_struggle', 'Difficulty in verifying the legitimacy of the charity or organization'),
    ('confidence_transparency_struggle', "Lack of transparency in the charity's financial reporting"),
    ('confidence_choice_struggle', 'Difficulty deciding which organization/cause or charity to support'),
    ('confidence_process_struggle', 'The process of making a donation (e.g., payment methods, paperwork)'),
]:
    add(jtbd_cfg(feature_id, 'confidence_risk', 'struggle', 'binary', 'Q7.2', Q_CHALLENGES, item,
                 adjustment='within_binary_question'))

# 10. Impact witness and feedback
add(jtbd_cfg('witness_communication_info', 'impact_witness', 'core_score', 'binary', 'Q6.2', Q_RESEARCH_INFO,
             'Methods of communication and donor engagement', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('witness_feedback_struggle', 'impact_witness', 'core_score', 'binary', 'Q7.2', Q_CHALLENGES,
             'Lack of feedback or results from previous donations', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('witness_story_message', 'impact_witness', 'core_score', 'likert', 'Q8.1', Q_MESSAGING,
             'Seeing personal stories or testimonials from those who have benefited from the charity', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
add(jtbd_cfg('witness_specific_impact_diagnostic', 'impact_witness', 'validator', 'likert', 'Q8.1', Q_MESSAGING,
             'Understanding the specific impact of your donation (e.g., how funds are used, success stories)',
             adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP',
             notes='Primary score home remains visible impact; this is a diagnostic only.'))

# 11. Active participation
add(jtbd_cfg('participation_importance', 'active_participation', 'core_score', 'likert', 'Q3.1', Q_IMPORTANCE,
             'Opportunities for personal involvement or volunteering with the organization', factor_pool=True,
             adjustment='within_ordered_battery', mapping_name='IMPORTANT_MAP'))
add(jtbd_cfg('participation_event_motivation', 'active_participation', 'core_score', 'binary', 'Q4.1', Q_MOTIVATIONS,
             'Participation in events or fundraisers', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('participation_volunteer_info', 'active_participation', 'core_score', 'binary', 'Q6.2', Q_RESEARCH_INFO,
             'Volunteer opportunities and ways to get involved', factor_pool=True, adjustment='within_binary_question'))
add(jtbd_cfg('participation_firsthand_message', 'active_participation', 'core_score', 'likert', 'Q8.1', Q_MESSAGING,
             "Opportunities to see or participate in the charity's work firsthand (e.g., site visits, volunteer opportunities)",
             factor_pool=True, adjustment='within_ordered_battery', mapping_name='INFLUENTIAL_MAP'))
for feature_id, question, item in [
    ('participation_prior_volunteer', Q_PRIOR_ACTIONS, 'Volunteered for community service'),
    ('participation_civic_activity', Q_ACTIVITIES, 'Volunteering for charitable or civic causes'),
    ('participation_classroom_volunteer', Q_SCHOOL, 'Volunteering in the classroom'),
    ('participation_school_fundraiser', Q_SCHOOL, 'Participating in school fundraisers'),
]:
    add(jtbd_cfg(feature_id, 'active_participation', 'validator', 'binary', 'validator', question, item,
                 adjustment='within_binary_question'))

# Education outcomes: expressions of general jobs, not job-score or factor inputs.
add(jtbd_cfg('edu_current_giver', None, 'education_outcome', 'binary', 'Q4.3', Q_CURRENT_CAUSES,
             'Educational-focused groups', notes='Current education giving.'))
add(jtbd_cfg('edu_open_ordinal', None, 'education_outcome', 'likert', 'Q4.5', Q_OPENNESS,
             'Educational-focused groups', mapping_name='OPEN_MAP',
             notes='Available only among respondents who do not currently give to education.'))
for feature_id, item in [
    ('edu_direct_teacher', "Direct donation to specific teachers' classrooms"),
    ('edu_after_school', 'Donations to after-school or extracurricular programs'),
    ('edu_scholarship', 'Contributions to scholarship funds'),
    ('edu_higher_education', 'Donations to higher education institutions (e.g., universities, colleges)'),
    ('edu_vocational_adult', 'Contributions to adult education or vocational training programs'),
    ('edu_none', 'I would not be likely to make any education-related donations'),
]:
    add(jtbd_cfg(feature_id, None, 'education_outcome', 'binary', 'Q4.6', Q_EDU_FORMATS, item))
add(jtbd_cfg('teacher_directed_preference', None, 'education_outcome', 'binary', 'Q4.6', Q_EDU_FORMATS,
             "Direct donation to specific teachers' classrooms",
             notes='Single-item proxy for the teacher-directed judgment hypothesis; not a validated composite job score.'))

jtbd_feature_config = pd.DataFrame(rows)

# Strong validation: one active primary home, exact columns, and no education/source/legacy leakage into core jobs.
assert jtbd_feature_config['feature_id'].is_unique
assert set(jtbd_feature_config['role']).issubset({
    'core_score', 'hiring_criterion', 'struggle', 'validator', 'activation', 'education_outcome', 'profile_only', 'exclude'
})
core_cfg = jtbd_feature_config[jtbd_feature_config['role'].eq('core_score')].copy()
assert set(core_cfg['job_family']) == set(JTBD_JOBS)
assert not core_cfg['source_column'].duplicated().any(), 'A raw item has more than one active score home.'
assert core_cfg['factor_pool'].all(), 'Every eligible mapped core item should enter the mixed-item latent pass.'
assert not core_cfg['source_column'].isin({SOURCE_COL, HISTORY_COL, LEGACY_CLUSTER_COL}).any()
assert not core_cfg['source_question'].isin({Q_CURRENT_CAUSES, Q_OPENNESS, Q_EDU_FORMATS}).any()
missing_cfg_columns = set(jtbd_feature_config['source_column']).difference(raw.columns)
assert not missing_cfg_columns, f'Configuration columns missing from export: {sorted(missing_cfg_columns)}'

JTBD_MAPPING_LOOKUP = {
    'IMPORTANT_MAP': IMPORTANT_MAP,
    'INFLUENTIAL_MAP': INFLUENTIAL_MAP,
    'AGREE_MAP': AGREE_MAP,
    'OPEN_MAP': OPEN_MAP,
    'FREQUENCY_MAP': FREQUENCY_MAP,
}


def jtbd_fixed_scale(series: pd.Series, mapping: Mapping[str, float]) -> pd.Series:
    values = series.map(mapping).astype(float)
    low = float(min(mapping.values()))
    high = float(max(mapping.values()))
    return (values - low) / (high - low)


def jtbd_binary_question_columns(question: str) -> list[str]:
    prefix = f'{question} - '
    columns = []
    for column in raw.columns:
        if not column.startswith(prefix):
            continue
        option = column[len(prefix):]
        if option == 'Other' or 'Text_' in option:
            continue
        columns.append(column)
    if not columns:
        raise KeyError(f'No binary columns found for {question!r}')
    return columns


def jtbd_ordered_battery_columns(question: str, mapping: Mapping[str, float]) -> list[str]:
    suffix = f' - {question}'
    columns = []
    for column in raw.columns:
        if not column.endswith(suffix):
            continue
        if column == QC_COL:
            continue
        observed = set(raw[column].dropna().unique())
        if observed and observed.issubset(set(mapping)):
            columns.append(column)
    if not columns:
        raise KeyError(f'No ordered columns found for {question!r}')
    return columns


def jtbd_derived_feature(frame: pd.DataFrame, feature_id: str) -> pd.Series:
    decision = frame[DECISION_COL]
    recipient_selected = decision.isin([
        'I knew exactly what cause/organization I wanted to donate to and the amount',
        'I had a general idea of what cause/organization I wanted to donate to but not the amount',
    ])
    amount_selected = decision.isin([
        'I knew exactly what cause/organization I wanted to donate to and the amount',
        'I had a general idea of the amount I wanted to donate but not the cause/organization',
    ])
    if feature_id == 'control_recipient_preselected':
        return recipient_selected.astype(float)
    if feature_id == 'control_amount_preselected':
        return amount_selected.astype(float)
    raise KeyError(feature_id)


def jtbd_feature_values(frame: pd.DataFrame, row: pd.Series) -> tuple[pd.Series, pd.Series]:
    response_type = row['response_type']
    source_column = row['source_column']
    adjustment = row['adjustment']

    if response_type == 'likert':
        mapping = JTBD_MAPPING_LOOKUP[row['mapping_name']]
        raw_value = jtbd_fixed_scale(frame[source_column], mapping)
    elif response_type == 'binary':
        raw_value = frame[source_column].notna().astype(float)
    elif response_type == 'frequency':
        raw_value = jtbd_fixed_scale(frame[source_column], FREQUENCY_MAP)
    elif response_type == 'derived':
        raw_value = jtbd_derived_feature(frame, row['feature_id'])
    elif response_type == 'single_choice':
        raise NotImplementedError('No single-choice core feature is currently configured.')
    else:
        raise ValueError(response_type)

    if adjustment == 'within_ordered_battery':
        mapping = JTBD_MAPPING_LOOKUP[row['mapping_name']]
        battery_columns = jtbd_ordered_battery_columns(row['source_question'], mapping)
        battery = frame[battery_columns].apply(lambda series: jtbd_fixed_scale(series, mapping))
        adjusted = 0.5 + 0.5 * (raw_value - battery.mean(axis=1))
    elif adjustment == 'within_binary_question':
        question_columns = jtbd_binary_question_columns(row['source_question'])
        selection_rate = frame[question_columns].notna().mean(axis=1)
        adjusted = 0.5 + 0.5 * (raw_value - selection_rate)
    elif adjustment in {'raw_anchor', 'none'}:
        adjusted = raw_value.copy()
    else:
        raise ValueError(f'Unknown adjustment={adjustment!r}')

    return raw_value.clip(0, 1), adjusted.clip(0, 1)


def jtbd_build_item_matrices(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_items = pd.DataFrame(index=frame.index)
    adjusted_items = pd.DataFrame(index=frame.index)
    for _, row in core_cfg.iterrows():
        raw_value, adjusted_value = jtbd_feature_values(frame, row)
        raw_items[row['feature_id']] = raw_value
        adjusted_items[row['feature_id']] = adjusted_value
    return raw_items, adjusted_items


def jtbd_build_scores(
    raw_items: pd.DataFrame,
    adjusted_items: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scores = pd.DataFrame(index=raw_items.index)
    contributions = []
    definitions = []
    for job_family, job_description in JTBD_JOBS.items():
        job_cfg = core_cfg[core_cfg['job_family'].eq(job_family)].copy()
        raw_question_parts = {}
        adjusted_question_parts = {}
        for question_group, question_cfg in job_cfg.groupby('question_group', sort=False):
            features = question_cfg['feature_id'].tolist()
            raw_question_parts[question_group] = raw_items[features].mean(axis=1, skipna=True)
            adjusted_question_parts[question_group] = adjusted_items[features].mean(axis=1, skipna=True)
            contributions.append(pd.DataFrame({
                ID_COL: jtbd_retained.loc[raw_items.index, ID_COL].to_numpy(),
                'job_family': job_family,
                'question_group': question_group,
                'raw_contribution': raw_question_parts[question_group].to_numpy(),
                'priority_contribution': adjusted_question_parts[question_group].to_numpy(),
            }))
        raw_questions = pd.DataFrame(raw_question_parts, index=raw_items.index)
        adjusted_questions = pd.DataFrame(adjusted_question_parts, index=raw_items.index)
        scores[f'absolute__{job_family}'] = raw_questions.mean(axis=1, skipna=True)
        scores[f'priority__{job_family}'] = adjusted_questions.mean(axis=1, skipna=True)
        definitions.append({
            'job_family': job_family,
            'job_description': job_description,
            'feature_n': len(job_cfg),
            'question_n': job_cfg['question_group'].nunique(),
            'features': ' | '.join(job_cfg['feature_id']),
            'questions': ' | '.join(job_cfg['question_group'].drop_duplicates()),
        })
    contribution_frame = pd.concat(contributions, ignore_index=True)
    return scores, contribution_frame, pd.DataFrame(definitions)

def jtbd_nearest_correlation(matrix: np.ndarray, floor: float = 1e-6) -> np.ndarray:
    symmetric = (matrix + matrix.T) / 2
    values, vectors = np.linalg.eigh(symmetric)
    values = np.clip(values, floor, None)
    psd = (vectors * values) @ vectors.T
    scale = np.sqrt(np.diag(psd))
    corr = psd / np.outer(scale, scale)
    np.fill_diagonal(corr, 1.0)
    return corr


def jtbd_spearman_corr(frame: pd.DataFrame) -> np.ndarray:
    corr = frame.corr(method='spearman').to_numpy(dtype=float)
    if np.isnan(corr).any():
        raise ValueError('Latent correlation matrix contains NaN values after eligibility filtering.')
    return jtbd_nearest_correlation(corr)


def jtbd_principal_axis(
    corr: np.ndarray,
    n_factors: int,
    max_iter: int = 200,
    tol: float = 1e-7,
) -> tuple[np.ndarray, np.ndarray]:
    inverse = np.linalg.pinv(corr)
    communalities = np.clip(1.0 - 1.0 / np.clip(np.diag(inverse), 1e-8, None), 0.05, 0.95)
    loadings = None
    for _ in range(max_iter):
        reduced = corr.copy()
        np.fill_diagonal(reduced, communalities)
        eigenvalues, eigenvectors = np.linalg.eigh(reduced)
        order = np.argsort(eigenvalues)[::-1][:n_factors]
        selected_values = np.clip(eigenvalues[order], 1e-8, None)
        loadings = eigenvectors[:, order] * np.sqrt(selected_values)
        updated = np.clip((loadings ** 2).sum(axis=1), 0.01, 0.999)
        if np.max(np.abs(updated - communalities)) < tol:
            communalities = updated
            break
        communalities = updated
    return loadings, communalities


def jtbd_varimax(loadings: np.ndarray, gamma: float = 1.0, q: int = 100, tol: float = 1e-7) -> tuple[np.ndarray, np.ndarray]:
    p, k = loadings.shape
    rotation = np.eye(k)
    previous = 0.0
    for _ in range(q):
        rotated = loadings @ rotation
        u, singular, vh = np.linalg.svd(
            loadings.T @ (rotated ** 3 - (gamma / p) * rotated @ np.diag(np.diag(rotated.T @ rotated)))
        )
        rotation = u @ vh
        current = singular.sum()
        if previous and current - previous < tol:
            break
        previous = current
    return loadings @ rotation, rotation


def jtbd_promax(loadings: np.ndarray, power: int = 4) -> tuple[np.ndarray, np.ndarray]:
    orthogonal, _ = jtbd_varimax(loadings)
    target = np.sign(orthogonal) * np.abs(orthogonal) ** power
    transform = np.linalg.lstsq(orthogonal, target, rcond=None)[0]
    norms = np.sqrt(np.diag(transform.T @ transform))
    norms = np.where(norms == 0, 1.0, norms)
    transform = transform / norms
    pattern = orthogonal @ transform
    phi = np.linalg.pinv(transform.T @ transform)
    return pattern, phi


def jtbd_factor_scores(frame: pd.DataFrame, corr: np.ndarray, loadings: np.ndarray) -> np.ndarray:
    ranked = frame.rank(axis=0, method='average', pct=True)
    standardized = StandardScaler().fit_transform(ranked)
    inverse = np.linalg.pinv(corr)
    weights = inverse @ loadings @ np.linalg.pinv(loadings.T @ inverse @ loadings)
    return standardized @ weights


def jtbd_parallel_analysis(frame: pd.DataFrame, runs: int, seed: int) -> pd.DataFrame:
    observed = np.linalg.eigvalsh(jtbd_spearman_corr(frame))[::-1]
    rng = np.random.default_rng(seed)
    random_values = np.zeros((runs, frame.shape[1]))
    array = frame.to_numpy(copy=True)
    for run in range(runs):
        shuffled = np.column_stack([rng.permutation(array[:, column]) for column in range(array.shape[1])])
        random_values[run] = np.linalg.eigvalsh(jtbd_spearman_corr(pd.DataFrame(shuffled)))[::-1]
    result = pd.DataFrame({
        'component': np.arange(1, frame.shape[1] + 1),
        'observed_eigenvalue': observed,
        'random_mean': random_values.mean(axis=0),
        'random_p95': np.quantile(random_values, 0.95, axis=0),
    })
    result['retained_by_p95'] = result['observed_eigenvalue'] > result['random_p95']
    return result


def jtbd_loading_congruence(reference: np.ndarray, candidate: np.ndarray) -> np.ndarray:
    output = np.zeros((reference.shape[1], candidate.shape[1]))
    for i in range(reference.shape[1]):
        for j in range(candidate.shape[1]):
            denominator = np.linalg.norm(reference[:, i]) * np.linalg.norm(candidate[:, j])
            output[i, j] = abs(float(reference[:, i] @ candidate[:, j]) / denominator) if denominator else 0.0
    return output


def jtbd_match_loadings(reference: np.ndarray, candidate: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    congruence = jtbd_loading_congruence(reference, candidate)
    rows, cols = linear_sum_assignment(-congruence)
    matched = candidate[:, cols].copy()
    values = congruence[rows, cols]
    for position, ref_factor in enumerate(rows):
        cand_factor = cols[position]
        sign = np.sign(reference[:, ref_factor] @ candidate[:, cand_factor]) or 1.0
        matched[:, ref_factor] = candidate[:, cand_factor] * sign
    return matched, values


def jtbd_latent_eligibility(frame: pd.DataFrame, config: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    eligible = []
    for feature_id in config['feature_id']:
        series = frame[feature_id]
        unique_n = series.nunique(dropna=True)
        sd = series.std()
        minimum_category_n = int(series.value_counts(dropna=False).min())
        reason = ''
        keep = True
        if unique_n < 2:
            keep, reason = False, 'constant'
        elif sd < 0.05:
            keep, reason = False, 'near_constant_sd'
        elif unique_n == 2 and minimum_category_n < 20:
            keep, reason = False, 'sparse_binary_category'
        rows.append({
            'feature_id': feature_id,
            'unique_n': unique_n,
            'sd': sd,
            'minimum_category_n': minimum_category_n,
            'eligible': keep,
            'exclusion_reason': reason,
        })
        if keep:
            eligible.append(feature_id)

    # Remove only deterministic duplicates, never merely high substantive correlations.
    if eligible:
        corr = frame[eligible].corr(method='spearman').abs()
        to_drop = set()
        for i, left in enumerate(eligible):
            if left in to_drop:
                continue
            for right in eligible[i + 1:]:
                if right not in to_drop and corr.loc[left, right] >= 0.999999:
                    to_drop.add(right)
        if to_drop:
            for row in rows:
                if row['feature_id'] in to_drop:
                    row['eligible'] = False
                    row['exclusion_reason'] = 'deterministic_duplicate'
            eligible = [feature for feature in eligible if feature not in to_drop]
    return pd.DataFrame(rows), frame[eligible].copy()


def jtbd_fit_latent(frame: pd.DataFrame, n_factors: int) -> dict[str, Any]:
    corr = jtbd_spearman_corr(frame)
    unrotated, communalities = jtbd_principal_axis(corr, n_factors)
    loadings, phi = jtbd_promax(unrotated)
    scores = jtbd_factor_scores(frame, corr, loadings)
    return {
        'corr': corr,
        'loadings': loadings,
        'phi': phi,
        'communalities': communalities,
        'scores': scores,
    }


def jtbd_factor_stability(
    frame: pd.DataFrame,
    reference_loadings: np.ndarray,
    n_factors: int,
    runs: int,
    fraction: float,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sample_n = int(round(len(frame) * fraction))
    rows = []
    for run in range(runs):
        sample_positions = np.sort(rng.choice(len(frame), size=sample_n, replace=False))
        sampled = frame.iloc[sample_positions]
        try:
            candidate = jtbd_fit_latent(sampled, n_factors)['loadings']
            _, congruence = jtbd_match_loadings(reference_loadings, candidate)
            for factor_index, value in enumerate(congruence, start=1):
                rows.append({'run': run, 'factor': factor_index, 'congruence': value, 'status': 'ok'})
        except Exception as exc:
            rows.append({'run': run, 'factor': np.nan, 'congruence': np.nan, 'status': type(exc).__name__})
    return pd.DataFrame(rows)

def jtbd_build_education_outcomes(frame: pd.DataFrame) -> pd.DataFrame:
    outcomes = pd.DataFrame(index=frame.index)
    outcomes['current_education_giver'] = selected(frame, Q_CURRENT_CAUSES, 'Educational-focused groups').astype(int)
    openness_raw = frame[likert_col('Educational-focused groups', Q_OPENNESS)].map(OPEN_MAP)
    outcomes['education_open_ordinal'] = openness_raw
    outcomes['education_open_moderate_plus'] = openness_raw.ge(3).where(openness_raw.notna())
    outcomes['direct_teacher'] = selected(frame, Q_EDU_FORMATS, "Direct donation to specific teachers' classrooms").astype(int)
    outcomes['after_school'] = selected(frame, Q_EDU_FORMATS, 'Donations to after-school or extracurricular programs').astype(int)
    outcomes['scholarship'] = selected(frame, Q_EDU_FORMATS, 'Contributions to scholarship funds').astype(int)
    outcomes['higher_education'] = selected(frame, Q_EDU_FORMATS, 'Donations to higher education institutions (e.g., universities, colleges)').astype(int)
    outcomes['vocational_adult'] = selected(frame, Q_EDU_FORMATS, 'Contributions to adult education or vocational training programs').astype(int)
    outcomes['no_education_interest'] = selected(frame, Q_EDU_FORMATS, 'I would not be likely to make any education-related donations').astype(int)
    return outcomes


def jtbd_demographic_matrix(frame: pd.DataFrame) -> pd.DataFrame:
    demo = pd.DataFrame(index=frame.index)
    demo['age'] = pd.to_numeric(frame[AGE_COL], errors='coerce')
    demo['children_at_home'] = (
        selected(frame, Q_CHILDREN_HOME, 'Children 12 years old or younger') |
        selected(frame, Q_CHILDREN_HOME, 'Children between 13-18 years old')
    ).astype(int)
    categories = pd.get_dummies(
        frame[[GENDER_COL, INCOME_COL]].fillna('Missing').astype(str),
        prefix=['gender', 'income'],
        drop_first=True,
        dtype=float,
    )
    demo = pd.concat([demo, categories], axis=1)
    demo['age'] = demo['age'].fillna(demo['age'].median())
    age_sd = demo['age'].std()
    demo['age'] = (demo['age'] - demo['age'].mean()) / (age_sd if age_sd else 1.0)
    return demo.astype(float)


def jtbd_standardize_completed_scores(scores: pd.DataFrame) -> pd.DataFrame:
    means = scores.mean(axis=0)
    sds = scores.std(axis=0).replace(0, 1.0)
    return (scores - means) / sds


def jtbd_fit_logistic(X: pd.DataFrame, y: pd.Series, seed: int) -> LogisticRegression:
    model = LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='liblinear',
        max_iter=2000,
        random_state=seed,
    )
    model.fit(X, y.astype(int))
    return model


def jtbd_top_quartile_lift(score: pd.Series, outcome: pd.Series) -> float:
    threshold = score.quantile(0.75)
    overall = outcome.mean()
    top = outcome.loc[score.ge(threshold)].mean()
    return float(top / overall) if overall > 0 else np.nan


def jtbd_bootstrap_coefficients(
    X: pd.DataFrame,
    y: pd.Series,
    job_columns: list[str],
    runs: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    n = len(y)
    for run in range(runs):
        positions = rng.integers(0, n, size=n)
        y_boot = y.iloc[positions]
        if y_boot.nunique() < 2:
            continue
        X_boot = X.iloc[positions]
        model = jtbd_fit_logistic(X_boot, y_boot, seed + run)
        coefficients = pd.Series(model.coef_[0], index=X.columns)
        for job_column in job_columns:
            rows.append({'run': run, 'job_family': job_column, 'coefficient': coefficients[job_column]})
    return pd.DataFrame(rows)

def empirical_percentiles(reference: pd.Series, values: pd.Series) -> np.ndarray:
    reference_values = np.sort(pd.to_numeric(reference, errors='coerce').dropna().to_numpy(dtype=float))
    if not len(reference_values):
        raise ValueError('Cannot rank against an empty reference distribution.')
    numeric = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    output = np.full(len(numeric), 0.5, dtype=float)
    valid = np.isfinite(numeric)
    left = np.searchsorted(reference_values, numeric[valid], side='left')
    right = np.searchsorted(reference_values, numeric[valid], side='right')
    output[valid] = (left + right + 1) / (2 * len(reference_values))
    return output


def project_factor_scores(
    prospect_items: pd.DataFrame,
    all_items: pd.DataFrame,
    corr: np.ndarray,
    loadings: np.ndarray,
) -> pd.DataFrame:
    prospect_ranked = prospect_items.rank(axis=0, method='average', pct=True)
    all_ranked = pd.DataFrame({
        column: empirical_percentiles(prospect_items[column], all_items[column])
        for column in prospect_items.columns
    }, index=all_items.index)
    scaler = StandardScaler().fit(prospect_ranked)
    inverse = np.linalg.pinv(corr)
    weights = inverse @ loadings @ np.linalg.pinv(loadings.T @ inverse @ loadings)
    scores = scaler.transform(all_ranked) @ weights
    return pd.DataFrame(scores, index=all_items.index, columns=[f"Factor {i}" for i in range(1, loadings.shape[1] + 1)])


def run_jtbd_measurement(
    *,
    factor_count: int,
    parallel_runs: int,
    stability_runs: int,
    bootstrap_runs: int,
    output_dir: Path,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_items, priority_items = jtbd_build_item_matrices(jtbd_retained)
    scores, contributions, score_definitions = jtbd_build_scores(raw_items, priority_items)
    score_table = jtbd_retained[[ID_COL, SOURCE_COL, HISTORY_COL]].copy()
    for column in scores:
        score_table[column] = scores[column]

    prospect_index = jtbd_prospects.index
    latent_cfg = core_cfg[core_cfg['factor_pool']].copy()
    latent_audit, latent_raw = jtbd_latent_eligibility(raw_items.loc[prospect_index, latent_cfg.feature_id], latent_cfg)
    eligible_features = latent_raw.columns.tolist()
    latent_priority = priority_items.loc[prospect_index, eligible_features]

    parallel = jtbd_parallel_analysis(latent_raw, parallel_runs, JTBD_SEED)
    raw_fit = jtbd_fit_latent(latent_raw, factor_count)
    priority_fit = jtbd_fit_latent(latent_priority, factor_count)
    matched_priority_loadings, loading_congruence = jtbd_match_loadings(raw_fit['loadings'], priority_fit['loadings'])
    factor_names = [f'Factor {index}' for index in range(1, factor_count + 1)]
    raw_loadings = pd.DataFrame(raw_fit['loadings'], index=eligible_features, columns=factor_names)
    priority_loadings = pd.DataFrame(matched_priority_loadings, index=eligible_features, columns=factor_names)

    stability_raw = jtbd_factor_stability(
        latent_raw, raw_fit['loadings'], factor_count, stability_runs,
        JTBD_STABILITY_FRACTION, JTBD_SEED + 1,
    )
    stability_summary = (
        stability_raw[stability_raw.status.eq('ok')]
        .groupby('factor').congruence.agg(['count', 'median', 'mean', 'min'])
    )

    education = jtbd_build_education_outcomes(jtbd_retained)
    demographics = jtbd_demographic_matrix(jtbd_prospects)
    outcome_names = [
        'current_education_giver', 'education_open_moderate_plus', 'direct_teacher',
        'after_school', 'scholarship', 'higher_education', 'vocational_adult',
        'no_education_interest',
    ]
    onramp_rows, bootstrap_frames = [], []
    for variant, prefix in [('absolute', 'absolute__'), ('priority', 'priority__')]:
        score_frame = score_table.loc[prospect_index, [f'{prefix}{job}' for job in JTBD_JOBS]].copy()
        score_frame.columns = list(JTBD_JOBS)
        standardized = jtbd_standardize_completed_scores(score_frame)
        for outcome_name in outcome_names:
            outcome = education.loc[prospect_index, outcome_name]
            available = outcome.notna()
            y = outcome.loc[available].astype(int)
            if y.nunique() < 2:
                continue
            X_jobs = standardized.loc[available]
            X_adjusted = pd.concat([X_jobs, demographics.loc[available]], axis=1)
            base_model = jtbd_fit_logistic(X_jobs, y, JTBD_SEED)
            adjusted_model = jtbd_fit_logistic(X_adjusted, y, JTBD_SEED)
            base_coef = pd.Series(base_model.coef_[0], index=X_jobs.columns)
            adjusted_coef = pd.Series(adjusted_model.coef_[0], index=X_adjusted.columns)
            boot = jtbd_bootstrap_coefficients(
                X_adjusted, y, list(JTBD_JOBS), bootstrap_runs,
                JTBD_SEED + 100 * (outcome_names.index(outcome_name) + 1) + (variant == 'priority'),
            )
            boot['score_variant'], boot['outcome'] = variant, outcome_name
            bootstrap_frames.append(boot)
            boot_summary = boot.groupby('job_family').coefficient.agg(
                bootstrap_median='median',
                bootstrap_low=lambda values: values.quantile(0.025),
                bootstrap_high=lambda values: values.quantile(0.975),
                sign_stability=lambda values: max((values > 0).mean(), (values < 0).mean()),
            )
            for job in JTBD_JOBS:
                onramp_rows.append({
                    'score_variant': variant, 'outcome': outcome_name, 'n': len(y),
                    'outcome_prevalence': y.mean(), 'job_family': job,
                    'base_standardized_coefficient': base_coef[job],
                    'demographic_adjusted_coefficient': adjusted_coef[job],
                    'top_quartile_lift': jtbd_top_quartile_lift(score_frame.loc[available, job], y),
                    **boot_summary.loc[job].to_dict(),
                })

    onramps = pd.DataFrame(onramp_rows)
    bootstrap = pd.concat(bootstrap_frames, ignore_index=True)
    priority_onramps = onramps[onramps.score_variant.eq('priority')].copy()
    coefficient_matrix = priority_onramps.pivot(index='job_family', columns='outcome', values='demographic_adjusted_coefficient')
    lift_matrix = priority_onramps.pivot(index='job_family', columns='outcome', values='top_quartile_lift')

    outputs = {
        'feature_config': jtbd_feature_config,
        'score_definitions': score_definitions,
        'score_table': score_table,
        'question_contributions': contributions,
        'latent_audit': latent_audit,
        'parallel': parallel,
        'raw_loadings': raw_loadings,
        'priority_loadings': priority_loadings,
        'factor_stability_runs': stability_raw,
        'factor_stability_summary': stability_summary,
        'education_outcomes': education,
        'onramps': onramps,
        'onramp_bootstrap': bootstrap,
        'coefficient_matrix': coefficient_matrix,
        'lift_matrix': lift_matrix,
        'eligible_features': eligible_features,
        'latent_raw': latent_raw,
        'latent_priority': latent_priority,
        'raw_fit': raw_fit,
        'priority_fit': priority_fit,
        'loading_congruence': loading_congruence,
        'factor_names': factor_names,
    }
    csv_map = {
        'jtbd_feature_config.csv': jtbd_feature_config,
        'jtbd_score_definitions.csv': score_definitions,
        'jtbd_respondent_scores.csv': score_table,
        'jtbd_question_contributions.csv': contributions,
        'jtbd_latent_item_eligibility.csv': latent_audit,
        'jtbd_parallel_analysis.csv': parallel,
        'jtbd_raw_factor_loadings.csv': raw_loadings,
        'jtbd_priority_factor_loadings.csv': priority_loadings,
        'jtbd_factor_stability_runs.csv': stability_raw,
        'jtbd_factor_stability_summary.csv': stability_summary,
        'jtbd_education_outcomes.csv': education,
        'jtbd_education_onramps.csv': onramps,
        'jtbd_onramp_bootstrap.csv': bootstrap,
        'jtbd_priority_coefficients.csv': coefficient_matrix,
        'jtbd_priority_lifts.csv': lift_matrix,
    }
    for filename, frame in csv_map.items():
        frame.to_csv(output_dir / filename, index=filename not in {'jtbd_feature_config.csv', 'jtbd_score_definitions.csv', 'jtbd_question_contributions.csv', 'jtbd_latent_item_eligibility.csv', 'jtbd_parallel_analysis.csv', 'jtbd_factor_stability_runs.csv', 'jtbd_education_onramps.csv', 'jtbd_onramp_bootstrap.csv'})
    return outputs


In [10]:
retained_model = retained.reset_index(drop=False).rename(columns={'index': '_raw_index'})
prospect_fit_mask = retained_model[SOURCE_COL].eq('Panel').to_numpy()
donor_mask = retained_model[SOURCE_COL].eq('List').to_numpy()
prospect_positions = np.flatnonzero(prospect_fit_mask)
donor_positions = np.flatnonzero(donor_mask)

UNIT_DISTANCE_CACHE, distance_qa = build_unit_distance_cache(retained_model, ACTIVE_UNITS, prospect_fit_mask)
charity_features_by_id = CHARITY_RESPONDENT_FEATURES.set_index(ID_COL)
charity_model_features = retained_model[[ID_COL]].join(
    charity_features_by_id.drop(columns=SOURCE_COL), on=ID_COL, validate='many_to_one'
)
for feature_name, fixed_range in CHARITY_STRUCTURE_CLUSTER_FEATURES.items():
    register_external_numeric_unit(
        UNIT_DISTANCE_CACHE, feature_name, CHARITY_WRITEIN_BLOCK,
        charity_model_features[feature_name], prospect_fit_mask, fixed_range,
    )
prospect_ids_in_model_order = retained_model.loc[prospect_fit_mask, ID_COL]
charity_common_entity_presence_model = CHARITY_COMMON_ENTITY_PRESENCE.reindex(
    prospect_ids_in_model_order
).fillna(0).astype(float)


def original_five_point_columns(frame: pd.DataFrame) -> list[tuple[str, Mapping[str, int]]]:
    result = []
    for column in frame.columns:
        if column == QC_COL:
            continue
        observed = set(frame[column].dropna().unique())
        for mapping in (IMPORTANT_MAP, INFLUENTIAL_MAP, AGREE_MAP, OPEN_MAP):
            if observed and observed.issubset(mapping) and frame[column].nunique(dropna=True) > 1:
                result.append((column, mapping))
                break
    return result


five_point_fields = original_five_point_columns(raw)
likert_numeric = pd.DataFrame({column: raw[column].map(mapping) for column, mapping in five_point_fields}, index=raw.index)
raw_response_intensity = likert_numeric.mean(axis=1)
retained_response_intensity = raw_response_intensity.loc[retained_model['_raw_index']].to_numpy(dtype=float)

segment_guardrails = pd.DataFrame(index=retained_model.index)
segment_guardrails['age_55_plus'] = pd.to_numeric(retained_model[AGE_COL], errors='coerce').ge(55)
segment_guardrails['female'] = retained_model[GENDER_COL].eq('Female')
segment_guardrails['nonwhite'] = pd.concat([
    selected(retained_model, RACE_QUESTION, option)
    for option in (
        'American Indian or Alaska Native', 'Asian', 'Black or African American',
        'Latino/Hispanic/Latinx', 'Native Hawaiian or Other Pacific Islander', 'Other'
    )
], axis=1).any(axis=1)
segment_guardrails['income_100k_plus'] = retained_model[INCOME_COL].isin([
    '$100,000 to $124,999', '$125,000 to $149,999', '$150,000 to $174,999',
    '$175,000 to $199,999', '$200,000+'
])
segment_guardrails['four_year_degree_plus'] = retained_model[EDUCATION_COL].isin([
    '4-year degree', 'Some graduate school', 'Graduate degree'
])
segment_guardrails['retired'] = retained_model[EMPLOYMENT_COL].eq('Retired')
segment_guardrails['children_in_home'] = selected(retained_model, CHILDREN_QUESTION, 'Children 12 years old or younger') | selected(
    retained_model, CHILDREN_QUESTION, 'Children between 13-18 years old'
)
segment_guardrails['urban'] = retained_model[URBANICITY_COL].eq('Urban area')
segment_guardrails['rural'] = retained_model[URBANICITY_COL].eq('Rural area')
segment_guardrails = segment_guardrails.astype(float)

PROFILE_VALUES, PROFILE_METADATA = build_profile_registry(retained_model)
PROFILE_METADATA.to_csv(OUTPUT_DIR / 'profile_feature_dictionary.csv')
distance_qa.to_csv(OUTPUT_DIR / 'distance_unit_qa.csv', index=False)

ENGINE_SNAPSHOT = object_snapshot(
    retained_model=retained_model,
    prospect_fit_mask=prospect_fit_mask,
    donor_mask=donor_mask,
)


# Zone 1 — Fixed evidence
These workstreams establish the settled facts: source/history confounding, response-intensity artifacts, sample limits, broad donor-market findings, and the forensic PCA/KMeans reconstruction. They are not experiment knobs.


In [11]:
derived_columns = []
for column in raw.columns:
    if ' - ' in column:
        base, suffix = column.rsplit(' - ', 1)
        if base in raw.columns and suffix in DERIVED_SUFFIXES:
            derived_columns.append(column)
qc_derived = [column for column in derived_columns if column.startswith('For quality control purposes')]
modeling_duplicate_columns = [column for column in derived_columns if column not in qc_derived]
all_blank_columns = [column for column in raw.columns if raw[column].isna().all()]
genuine_sparse_options = [
    column for column in raw.columns
    if column not in derived_columns and 'Text_' not in column
    and raw[column].nunique(dropna=True) == 1
    and raw[column].notna().any() and raw[column].isna().any()
]
original_multi_value_columns = [column for column in raw.columns if raw[column].nunique(dropna=True) > 1]

structure_audit = pd.DataFrame([
    ('Respondents', raw.shape[0]),
    ('Exported fields', raw.shape[1]),
    ('Redundant derived fields excluding QC', len(modeling_duplicate_columns)),
    ('Original five-point items behind derivatives', len({column.rsplit(' - ', 1)[0] for column in modeling_duplicate_columns})),
    ('Genuine sparse select-all fields', len(genuine_sparse_options)),
    ('Original fields with multiple values', len(original_multi_value_columns)),
    ('Fully blank fields', len(all_blank_columns)),
], columns=['measure', 'value'])

population_table = pd.DataFrame([
    ('Full qualified file', len(raw)),
    ('Attention-check pass', len(retained)),
    ('Attention-check fail — excluded', int((~attention_pass).sum())),
    ('Retained panel prospects — fit population', int(prospect_mask_retained.sum())),
    ('Retained list/past donors — post-fit cohort', int(donor_mask_retained.sum())),
], columns=['population', 'n'])

source_history = pd.crosstab(raw[SOURCE_COL], raw[HISTORY_COL], margins=True)
source_by_legacy = pd.crosstab(raw[LEGACY_CLUSTER_COL], raw[SOURCE_COL], margins=True)
source_by_legacy['List share'] = source_by_legacy.get('List', 0) / source_by_legacy['All']
legacy_donor_evidence = pd.DataFrame({
    'measure': [
        'Past donors labeled Academic Advocates',
        'Academic Advocates who are past donors',
        'Attention-pass past donors labeled Academic Advocates',
    ],
    'value': [
        ((raw[SOURCE_COL].eq('List')) & raw[LEGACY_CLUSTER_COL].eq('Academic Advocates')).sum() / raw[SOURCE_COL].eq('List').sum(),
        ((raw[SOURCE_COL].eq('List')) & raw[LEGACY_CLUSTER_COL].eq('Academic Advocates')).sum() / raw[LEGACY_CLUSTER_COL].eq('Academic Advocates').sum(),
        ((retained[SOURCE_COL].eq('List')) & retained[LEGACY_CLUSTER_COL].eq('Academic Advocates')).sum() / retained[SOURCE_COL].eq('List').sum(),
    ],
})

structure_audit.to_csv(OUTPUT_DIR / 'evidence_structure_audit.csv', index=False)
population_table.to_csv(OUTPUT_DIR / 'evidence_population_counts.csv', index=False)
source_history.to_csv(OUTPUT_DIR / 'evidence_source_history_equivalence.csv')
source_by_legacy.to_csv(OUTPUT_DIR / 'evidence_source_by_legacy_cluster.csv')
legacy_donor_evidence.to_csv(OUTPUT_DIR / 'evidence_legacy_donor_confound.csv', index=False)

print('FIXED EVIDENCE — DATA STRUCTURE AND CONFOUND')
print_decision_table(population_table)
print('\nLegacy donor confound:')
print(legacy_donor_evidence.assign(value=legacy_donor_evidence.value.map(lambda x: f'{x:.1%}')).to_string(index=False))


FIXED EVIDENCE — DATA STRUCTURE AND CONFOUND
                                 population    n
                        Full qualified file 1438
                       Attention-check pass 1245
            Attention-check fail — excluded  193
  Retained panel prospects — fit population 1120
Retained list/past donors — post-fit cohort  125

Legacy donor confound:
                                              measure value
               Past donors labeled Academic Advocates 95.7%
               Academic Advocates who are past donors 73.0%
Attention-pass past donors labeled Academic Advocates 95.2%


In [12]:
response_intensity_table = (
    pd.DataFrame({'legacy_cluster': raw[LEGACY_CLUSTER_COL], 'mean_likert': raw_response_intensity})
    .groupby('legacy_cluster').agg(n=('mean_likert', 'size'), mean_likert=('mean_likert', 'mean'))
    .sort_values('mean_likert')
)

sample_limits = pd.DataFrame({
    'measure': ['Recent charitable donor by screener', 'Income below $60,000', 'Age 55+', 'Retired', 'Four-year degree+'],
    'share': [
        1.0,
        raw[INCOME_COL].isin(['$0 to $24,999', '$25,000 to $59,999']).mean(),
        pd.to_numeric(raw[AGE_COL], errors='coerce').ge(55).mean(),
        raw[EMPLOYMENT_COL].eq('Retired').mean(),
        raw[EDUCATION_COL].isin(['4-year degree', 'Some graduate school', 'Graduate degree']).mean(),
    ],
})

analysis = retained
trust_items = {
    'Effectiveness and impact': "Effectiveness and impact of the organization's programs",
    'Transparency and accountability': 'Transparency and accountability of the organization',
    'Efficiency / low administrative cost': 'Efficiency in using donations (low administrative costs)',
    'Security and ease': 'Security and ease of the donation process',
}
trust_table = pd.DataFrame([
    {'finding': name, 'share': analysis[likert_col(item, Q_IMPORTANCE)].isin(['Very important', 'Extremely important']).mean()}
    for name, item in trust_items.items()
])
value_table = pd.DataFrame([
    {'finding': name, 'share': selected(analysis, Q_MOTIVATIONS, option).mean()}
    for name, option in {
        'Mission and values': 'Belief in the mission and values of the organization',
        'Positive community impact': 'Desire to make a positive impact in the community',
        'Personal connection': 'Personal connection to the cause or issue',
        'Trust in effectiveness/transparency': 'Trust in the effectiveness and transparency of the organization',
    }.items()
] + [{'finding': 'Prefers local-community impact', 'share': selected(analysis, Q_IMPACT_SCALE, 'My local community').mean()}])

decision_counts = analysis[DECISION_COL].value_counts(normalize=True).rename_axis('decision_state').reset_index(name='share')
research_table = pd.DataFrame([
    {'finding': 'Researches thoroughly — agree/top-two', 'share': analysis[likert_col('I research an organization thoroughly before making a donation', Q_CHARITY_ATTITUDES)].isin(['Somewhat agree', 'Strongly agree']).mean()},
    {'finding': 'Sticks to trusted organizations — agree/top-two', 'share': analysis[likert_col('I tend to stick to organizations that I trust and prefer and do not seek to find new ones for potential donations', Q_CHARITY_ATTITUDES)].isin(['Somewhat agree', 'Strongly agree']).mean()},
])
friction_table = pd.DataFrame([
    {'finding': 'Concern about use of funds', 'share': selected(analysis, Q_CHALLENGES, 'Concerns about how my donation will be used').mean()},
    {'finding': 'Uncertain effectiveness', 'share': selected(analysis, Q_CHALLENGES, 'Uncertainty about the effectiveness of my donation in making a real difference').mean()},
    {'finding': 'Transparency concern', 'share': selected(analysis, Q_CHALLENGES, "Lack of transparency in the charity's financial reporting").mean()},
    {'finding': 'Specific-impact message influential', 'share': analysis[likert_col('Understanding the specific impact of your donation (e.g., how funds are used, success stories)', Q_MESSAGING)].isin(['Very influential', 'Extremely influential']).mean()},
    {'finding': 'Transparency message influential', 'share': analysis[likert_col("Information about the charity's transparency and financial accountability", Q_MESSAGING)].isin(['Very influential', 'Extremely influential']).mean()},
])
education_potential = pd.DataFrame([
    {'finding': 'Currently gives to education', 'share': selected(analysis, Q_CURRENT_CAUSES, 'Educational-focused groups').mean()},
    {'finding': 'Direct teacher/classroom preference', 'share': selected(analysis, Q_EDU_FORMATS, "Direct donation to specific teachers' classrooms").mean()},
    {'finding': 'After-school preference', 'share': selected(analysis, Q_EDU_FORMATS, 'Donations to after-school or extracurricular programs').mean()},
    {'finding': 'Rejects education giving', 'share': selected(analysis, Q_EDU_FORMATS, 'I would not be likely to make any education-related donations').mean()},
])

for filename, frame in {
    'evidence_response_intensity.csv': response_intensity_table,
    'evidence_sample_limits.csv': sample_limits,
    'evidence_trust.csv': trust_table,
    'evidence_values_local.csv': value_table,
    'evidence_decision_state.csv': decision_counts,
    'evidence_research.csv': research_table,
    'evidence_friction_messages.csv': friction_table,
    'evidence_education_potential.csv': education_potential,
}.items():
    frame.to_csv(OUTPUT_DIR / filename, index=not isinstance(frame.index, pd.RangeIndex))

fixed_market_summary = pd.concat([
    trust_table.assign(section='Trust/proof'), value_table.assign(section='Values/local'),
    research_table.assign(section='Research'), friction_table.assign(section='Friction/message'),
    education_potential.assign(section='Education potential'),
], ignore_index=True)[['section', 'finding', 'share']]
print('FIXED EVIDENCE — MARKET SUMMARY')
print(fixed_market_summary.assign(share=fixed_market_summary.share.map(lambda x: f'{x:.1%}')).to_string(index=False))


FIXED EVIDENCE — MARKET SUMMARY
            section                                         finding share
        Trust/proof                        Effectiveness and impact 85.9%
        Trust/proof                 Transparency and accountability 84.9%
        Trust/proof            Efficiency / low administrative cost 77.9%
        Trust/proof                               Security and ease 77.1%
       Values/local                              Mission and values 66.9%
       Values/local                       Positive community impact 54.1%
       Values/local                             Personal connection 52.7%
       Values/local             Trust in effectiveness/transparency 53.2%
       Values/local                  Prefers local-community impact 59.0%
           Research           Researches thoroughly — agree/top-two 68.0%
           Research Sticks to trusted organizations — agree/top-two 72.2%
   Friction/message                      Concern about use of funds 41.9%
   Fri

In [13]:
FORENSIC_COLUMNS = {
    'involvement': likert_col('Opportunities for personal involvement or volunteering with the organization', Q_IMPORTANCE),
    'recognition': likert_col('Recognition or acknowledgment of donors', Q_IMPORTANCE),
    'tax': likert_col('I am motivated to donate by tax benefits', Q_CHARITY_ATTITUDES),
    'moral': likert_col('I donate to charity out of a moral/religious obligation', Q_CHARITY_ATTITUDES),
    'history': HISTORY_COL,
    'education': option_col(Q_CURRENT_CAUSES, 'Educational-focused groups'),
    'teacher_requests': option_col(Q_DISCOVERY, 'Direct requests from community figures (e.g., teachers, organizers)'),
    'emotion': option_col(Q_MOTIVATIONS, 'Emotional response to a specific event or campaign'),
    'social_ads': option_col(Q_AD_RESPONSE, 'Ads embedded in social media apps'),
}

forensic = pd.DataFrame(index=retained.index)
forensic['involvement'] = retained[FORENSIC_COLUMNS['involvement']].map(IMPORTANT_MAP)
forensic['recognition'] = retained[FORENSIC_COLUMNS['recognition']].map(IMPORTANT_MAP)
forensic['tax'] = retained[FORENSIC_COLUMNS['tax']].map(AGREE_MAP)
forensic['moral'] = retained[FORENSIC_COLUMNS['moral']].map(AGREE_MAP)
forensic['history'] = retained[HISTORY_COL].eq('Past donor (List)').astype(int)
for key in ['education', 'teacher_requests', 'emotion', 'social_ads']:
    forensic[key] = retained[FORENSIC_COLUMNS[key]].notna().astype(int)
assert forensic.notna().all().all()

def varimax(loadings: np.ndarray, gamma: float = 1.0, q: int = 100, tol: float = 1e-7) -> tuple[np.ndarray, np.ndarray]:
    p, k = loadings.shape
    rotation = np.eye(k)
    objective = 0.0
    for _ in range(q):
        old_objective = objective
        rotated = loadings @ rotation
        u, singular_values, vh = np.linalg.svd(
            loadings.T @ (rotated ** 3 - (gamma / p) * rotated @ np.diag(np.diag(rotated.T @ rotated)))
        )
        rotation = u @ vh
        objective = singular_values.sum()
        if old_objective and objective / old_objective < 1 + tol:
            break
    return loadings @ rotation, rotation

@dataclass
class ForensicRun:
    name: str
    labels: np.ndarray
    scores: np.ndarray
    loadings: pd.DataFrame
    explained_variance_ratio: np.ndarray


def forensic_run(
    X: pd.DataFrame,
    name: str,
    fit_mask: np.ndarray | None = None,
    n_components: int = 3,
) -> ForensicRun:
    if fit_mask is None:
        fit_mask = np.ones(len(X), dtype=bool)
    scaler = StandardScaler().fit(X.loc[fit_mask])
    z_fit = scaler.transform(X.loc[fit_mask])
    z_all = scaler.transform(X)
    pca = PCA(n_components=n_components, random_state=SEED).fit(z_fit)
    score_fit = pca.transform(z_fit)
    score_all = pca.transform(z_all)
    raw_loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    rotated_loadings, rotation = varimax(raw_loadings)
    score_fit = score_fit @ rotation
    score_all = score_all @ rotation
    kmeans = KMeans(n_clusters=4, n_init=100, random_state=SEED).fit(score_fit)
    labels = kmeans.predict(score_all)
    return ForensicRun(
        name=name,
        labels=labels,
        scores=score_all,
        loadings=pd.DataFrame(rotated_loadings, index=X.columns, columns=['RC1', 'RC2', 'RC3']),
        explained_variance_ratio=pca.explained_variance_ratio_,
    )

run_a = forensic_run(forensic, 'A. Full nine-variable model')
run_b = forensic_run(forensic.drop(columns='history'), 'B. Remove donor history')
run_c = forensic_run(
    forensic.drop(columns='history'),
    'C. Fit prospects, then assign donors',
    fit_mask=retained[SOURCE_COL].eq('Panel').to_numpy(),
)

pca_loadings = pca_loadings_raw.iloc[:9, :4].copy()
pca_loadings.columns = ['variable', 'RC1', 'RC2', 'RC3']
pca_loadings = pca_loadings.set_index('variable').apply(pd.to_numeric)
reported_rc2 = pca_loadings.loc[
    [
        HISTORY_COL,
        option_col(Q_CURRENT_CAUSES, 'Educational-focused groups'),
        option_col(Q_DISCOVERY, 'Direct requests from community figures (e.g., teachers, organizers)'),
    ],
    'RC2',
].rename(index={
    HISTORY_COL: 'DonorsChoose donation history',
    option_col(Q_CURRENT_CAUSES, 'Educational-focused groups'): 'Currently gives to education',
    option_col(Q_DISCOVERY, 'Direct requests from community figures (e.g., teachers, organizers)'): 'Teacher/community discovery',
})
reported_rc2_table = reported_rc2.to_frame('Reported RC2 loading')
reported_rc2_table.to_csv(OUTPUT_DIR / 'forensic_reported_education_loadings.csv')

# The reconstructed component that is most correlated with donor history.
history_correlations = [np.corrcoef(run_a.scores[:, i], forensic['history'])[0, 1] for i in range(3)]
history_component = int(np.argmax(np.abs(history_correlations)))
reconstructed_factor = run_a.loadings.iloc[:, history_component].sort_values(ascending=False).to_frame('Loading')
print(f'Reconstructed component most correlated with donor history: RC{history_component + 1}; r={history_correlations[history_component]:.3f}')

score_by_history = pd.DataFrame({
    'component_score': run_a.scores[:, history_component],
    'donor_history': np.where(forensic['history'].eq(1), 'Past donor/list', 'Panel prospect'),
})
score_summary = score_by_history.groupby('donor_history')['component_score'].agg(['count', 'mean', 'std'])
score_summary.to_csv(OUTPUT_DIR / 'forensic_component_by_history.csv')

def cluster_donor_summary(labels: np.ndarray) -> tuple[pd.DataFrame, int]:
    donor = forensic['history'].to_numpy().astype(bool)
    table = pd.crosstab(pd.Series(labels, name='cluster'), pd.Series(donor, name='past_donor'))
    for val in [False, True]:
        if val not in table.columns:
            table[val] = 0
    table = table.reindex(columns=[False, True], fill_value=0).rename(columns={False: 'prospects', True: 'past_donors'})
    table['cluster_n'] = table.sum(axis=1)
    table['donor_share_of_cluster'] = table['past_donors'] / table['cluster_n']
    table['share_of_all_donors'] = table['past_donors'] / table['past_donors'].sum()
    table['share_of_all_prospects'] = table['prospects'] / table['prospects'].sum()
    donor_heaviest_cluster = int(table['past_donors'].idxmax())
    return table, donor_heaviest_cluster

summary_a, donor_cluster_a = cluster_donor_summary(run_a.labels)
summary_b, donor_cluster_b = cluster_donor_summary(run_b.labels)
summary_c, donor_cluster_c = cluster_donor_summary(run_c.labels)

legacy_academic = retained[LEGACY_CLUSTER_COL].eq('Academic Advocates').to_numpy()
retained_donor = forensic['history'].to_numpy().astype(bool)

comparison = pd.DataFrame([
    {
        'run': run_a.name,
        'education/donor cluster n': int(summary_a.loc[donor_cluster_a, 'cluster_n']),
        'past donors in cluster': int(summary_a.loc[donor_cluster_a, 'past_donors']),
        'donor share of cluster': summary_a.loc[donor_cluster_a, 'donor_share_of_cluster'],
        'share of all past donors captured': summary_a.loc[donor_cluster_a, 'share_of_all_donors'],
        'prospect share in cluster': summary_a.loc[donor_cluster_a, 'share_of_all_prospects'],
        'legacy Academic Advocates in cluster': int((legacy_academic & (run_a.labels == donor_cluster_a)).sum()),
    },
    {
        'run': run_b.name,
        'education/donor cluster n': int(summary_b.loc[donor_cluster_b, 'cluster_n']),
        'past donors in cluster': int(summary_b.loc[donor_cluster_b, 'past_donors']),
        'donor share of cluster': summary_b.loc[donor_cluster_b, 'donor_share_of_cluster'],
        'share of all past donors captured': summary_b.loc[donor_cluster_b, 'share_of_all_donors'],
        'prospect share in cluster': summary_b.loc[donor_cluster_b, 'share_of_all_prospects'],
        'legacy Academic Advocates in cluster': int((legacy_academic & (run_b.labels == donor_cluster_b)).sum()),
    },
    {
        'run': run_c.name,
        'education/donor cluster n': int(summary_c.loc[donor_cluster_c, 'cluster_n']),
        'past donors in cluster': int(summary_c.loc[donor_cluster_c, 'past_donors']),
        'donor share of cluster': summary_c.loc[donor_cluster_c, 'donor_share_of_cluster'],
        'share of all past donors captured': summary_c.loc[donor_cluster_c, 'share_of_all_donors'],
        'prospect share in cluster': summary_c.loc[donor_cluster_c, 'share_of_all_prospects'],
        'legacy Academic Advocates in cluster': int((legacy_academic & (run_c.labels == donor_cluster_c)).sum()),
    },
]).set_index('run')

comparison.to_csv(OUTPUT_DIR / 'forensic_three_run_comparison.csv')

# Expected diagnostic landmarks from the frozen data and deterministic implementation.
assert comparison.loc[run_a.name, 'past donors in cluster'] == 116
assert math.isclose(comparison.loc[run_a.name, 'donor share of cluster'], 0.734177, rel_tol=1e-4)
assert math.isclose(comparison.loc[run_a.name, 'share of all past donors captured'], 0.928, rel_tol=1e-3)
assert math.isclose(comparison.loc[run_c.name, 'share of all past donors captured'], 0.784, rel_tol=1e-3)

legacy_academic_donors = retained[SOURCE_COL].eq('List') & retained[LEGACY_CLUSTER_COL].eq('Academic Advocates')
academic_donor_assignments = pd.Series(run_c.labels[legacy_academic_donors.to_numpy()], name='prospect_fitted_cluster')
migration = academic_donor_assignments.value_counts().sort_index().to_frame('legacy Academic past donors')
migration['share'] = migration['legacy Academic past donors'] / migration['legacy Academic past donors'].sum()
migration.to_csv(OUTPUT_DIR / 'forensic_legacy_academic_migration.csv')

outside_primary = int((academic_donor_assignments != donor_cluster_c).sum())
assert outside_primary == 21

print(
    f'Of the {legacy_academic_donors.sum()} attention-pass prior donors originally labeled Academic Advocates, '
    f'{outside_primary} ({outside_primary / legacy_academic_donors.sum():.1%}) map outside the prospect-derived education-oriented cluster.'
)

print('FIXED EVIDENCE — LEGACY FORENSIC COMPARISON')
print(comparison.to_string())


Reconstructed component most correlated with donor history: RC2; r=0.805
Of the 119 attention-pass prior donors originally labeled Academic Advocates, 21 (17.6%) map outside the prospect-derived education-oriented cluster.
FIXED EVIDENCE — LEGACY FORENSIC COMPARISON
                                      education/donor cluster n  past donors in cluster  donor share of cluster  share of all past donors captured  prospect share in cluster  legacy Academic Advocates in cluster
run                                                                                                                                                                                                                
A. Full nine-variable model                                 158                     116                   0.734                              0.928                      0.037                                   157
B. Remove donor history                                     302                      97          

# Zone 2 — Experiment surface
## How to run an experiment
Edit **only** the PARAMS panel for the question you are testing, then run that workstream downward. The engine and fixed-evidence cells should not be edited during a marketing working session.


## Loop 1 — Main segmentation experiment


In [162]:
# SEGMENTATION PARAMS — edit values only, then run this workstream downward.
SEGMENT_BLOCK_WEIGHTS = {
    'giving_job_impact': 2.0,
    'trust_research_friction': 1.0,
    'trigger_channel': 1.0,
    'giving_commitment': 0.5,
    'education_receptivity': 0.25,    # 0 removes education from cluster formation; it remains profile-only.
}
SEGMENT_UNIT_DOWNWEIGHTS = {
    'giving_attitudes': 0.25,
    'organization_selection_attributes': 0.25,
    'trigger_messages': 0.25,
    'research_and_incumbent_trust': 0.25,
    'organizations_per_year': 0.0,
    'current_education_giver': 0.0,
}
SEGMENT_K_RANGE = (2,3,4,5)            # Safe search: 3–6; current candidate is k=3.
SEGMENT_MIN_CLUSTER_SHARE = 0.07       # Reject smaller clusters unless separately approved with strong Jaccard.
SEGMENT_RESPONSE_STYLE_ADJUSTED = False
SEGMENT_RESAMPLES = '100'              # 'smoke', '20', '50', or '100'. Use 100 only for final sign-off.


In [163]:
assert SEGMENT_RESAMPLES in EXPERIMENT_MODES
assert all(2 <= value <= 6 for value in SEGMENT_K_RANGE)
assert set(SEGMENT_UNIT_DOWNWEIGHTS).issubset(ACTIVE_UNIT_NAMES)
active_blocks = tuple(block for block, weight in SEGMENT_BLOCK_WEIGHTS.items() if weight > 0)
working_config = SegmentExperiment(
    name='working_segmentation',
    include_blocks=active_blocks,
    block_weights={block: weight for block, weight in SEGMENT_BLOCK_WEIGHTS.items() if weight > 0},
    unit_weights=SEGMENT_UNIT_DOWNWEIGHTS,
    response_style_adjusted=SEGMENT_RESPONSE_STYLE_ADJUSTED,
    k_values=SEGMENT_K_RANGE,
    notes='Current collaborative working configuration; prospect-fit only.',
)
working_run = run_segment_experiment(working_config, UNIT_DISTANCE_CACHE, SEGMENT_RESAMPLES)
segmentation_decision_table = working_run.candidate_summary.reset_index().rename(columns={
    'mean_stability_ARI': 'mean_ARI',
    'weakest_cluster_median_jaccard': 'weakest_cluster_jaccard',
    'minimum_cluster_share': 'min_share',
})
segmentation_decision_table['passes_min_share'] = segmentation_decision_table.min_share.ge(SEGMENT_MIN_CLUSTER_SHARE)
segmentation_decision_table.to_csv(OUTPUT_DIR / 'segmentation_decision_table.csv', index=False)

print('SEGMENTATION DECISION TABLE')
print_decision_table(segmentation_decision_table, [
    'experiment', 'k', 'weakest_cluster_jaccard', 'mean_ARI', 'min_share',
    'silhouette', 'passes_min_share',
])


SEGMENTATION DECISION TABLE
          experiment  k  weakest_cluster_jaccard  mean_ARI  min_share  silhouette  passes_min_share
working_segmentation  2                    0.653     0.481      0.295       0.121              True
working_segmentation  3                    0.910     0.805      0.246       0.113              True
working_segmentation  4                    0.671     0.706      0.135       0.132              True
working_segmentation  5                    0.050     0.538      0.120       0.094              True


In [164]:
eligible_segmentation = segmentation_decision_table[segmentation_decision_table.passes_min_share].copy()
if eligible_segmentation.empty:
    raise ValueError('No segmentation candidate passes the minimum cluster-share gate.')
eligible_segmentation['stability_floor'] = eligible_segmentation[['mean_ARI', 'weakest_cluster_jaccard']].min(axis=1)
selected_segmentation_row = eligible_segmentation.sort_values(
    ['stability_floor', 'mean_ARI', 'weakest_cluster_jaccard', 'silhouette'], ascending=False
).iloc[0]
SELECTED_K = int(selected_segmentation_row.k)
selected_assignments = working_run.assignments[SELECTED_K]
selected_profile, selected_cluster_summary = profile_solution(working_run, SELECTED_K, PROFILE_VALUES, PROFILE_METADATA)
selected_profile.to_csv(OUTPUT_DIR / 'selected_segmentation_profile_tidy.csv', index=False)
selected_cluster_summary.to_csv(OUTPUT_DIR / 'selected_segmentation_cluster_summary.csv')
selected_assignments.to_csv(OUTPUT_DIR / 'selected_segmentation_assignments.csv', index=False)

# Education is deliberately an outcome/profile when its block weight is zero.
education_outcomes = jtbd_build_education_outcomes(retained_model)
education_outcomes['children_youth_giver'] = selected(retained_model, Q_CURRENT_CAUSES, "Children's charities and youth programs").astype(int)
education_outcomes['teacher_community_discovery'] = selected(retained_model, Q_DISCOVERY, 'Direct requests from community figures (e.g., teachers, organizers)').astype(int)
prospect_education = education_outcomes.loc[prospect_fit_mask].copy()
prospect_education['cluster'] = selected_assignments.loc[prospect_fit_mask, 'persona_cluster'].to_numpy()
education_profile = prospect_education.groupby('cluster').mean(numeric_only=True)
education_profile.loc['overall'] = prospect_education.drop(columns='cluster').mean(numeric_only=True)
education_profile.to_csv(OUTPUT_DIR / 'selected_segmentation_education_profile.csv')

# Rule-based charity intelligence stays post-fit/profile-only.
charity_profile = CHARITY_MENTIONS[CHARITY_MENTIONS.is_charitable_mention].merge(
    selected_assignments[[ID_COL, 'persona_cluster']], on=ID_COL, how='inner', validate='many_to_one'
)
charity_entity_profile = (
    charity_profile.groupby(['persona_cluster', 'canonical_entity']).size().rename('mention_n').reset_index()
)
charity_entity_profile.to_csv(OUTPUT_DIR / 'selected_segmentation_charity_entity_profile.csv', index=False)

print('\nSEGMENTATION DECISION CARD')
print(f'Params: blocks={SEGMENT_BLOCK_WEIGHTS}; k={SEGMENT_K_RANGE}; resamples={SEGMENT_RESAMPLES}')
print(f'Selected for inspection: k={SELECTED_K}')
print(f"Stability: ARI={selected_segmentation_row.mean_ARI:.3f}; weakest Jaccard={selected_segmentation_row.weakest_cluster_jaccard:.3f}")
print(f"Size gate: min share={selected_segmentation_row.min_share:.1%}; passed={bool(selected_segmentation_row.passes_min_share)}")
print('Recommendation: inspect actionability and education-on-ramp profiles; do not name/freeze before the shortlist gate.')



SEGMENTATION DECISION CARD
Params: blocks={'giving_job_impact': 2.0, 'trust_research_friction': 1.0, 'trigger_channel': 1.0, 'giving_commitment': 0.5, 'education_receptivity': 0.25}; k=(2, 3, 4, 5); resamples=100
Selected for inspection: k=3
Stability: ARI=0.805; weakest Jaccard=0.910
Size gate: min share=24.6%; passed=True
Recommendation: inspect actionability and education-on-ramp profiles; do not name/freeze before the shortlist gate.


## Loop 2 — JTBD measurement and education on-ramps


In [17]:
# JTBD PARAMS — edit values only, then run this workstream downward.
JTBD_FACTOR_COUNT_PARAM = 7             # Safe range 3–8; seven is the current validated interpretation.
JTBD_PARALLEL_RUNS_PARAM = 20           # Use 100 for final parallel-analysis sign-off.
JTBD_STABILITY_RUNS_PARAM = 20          # Use 100 for final factor-stability sign-off.
JTBD_BOOTSTRAP_RUNS_PARAM = 20          # Use 100 for final on-ramp intervals.


In [18]:
assert JTBD_FACTOR_MIN <= JTBD_FACTOR_COUNT_PARAM <= JTBD_FACTOR_MAX
JTBD_OUTPUT_DIR = OUTPUT_DIR / 'jtbd'
jtbd_results = run_jtbd_measurement(
    factor_count=JTBD_FACTOR_COUNT_PARAM,
    parallel_runs=JTBD_PARALLEL_RUNS_PARAM,
    stability_runs=JTBD_STABILITY_RUNS_PARAM,
    bootstrap_runs=JTBD_BOOTSTRAP_RUNS_PARAM,
    output_dir=JTBD_OUTPUT_DIR,
)

factor_stability = jtbd_results['factor_stability_summary'].reset_index()
factor_stability.to_csv(OUTPUT_DIR / 'jtbd_decision_table.csv', index=False)
print('JTBD DECISION TABLE')
print_decision_table(factor_stability, ['factor', 'median', 'mean', 'min'])

priority_onramps = jtbd_results['onramps'][jtbd_results['onramps'].score_variant.eq('priority')].copy()
direct_teacher = priority_onramps[priority_onramps.outcome.eq('direct_teacher')].sort_values(
    ['demographic_adjusted_coefficient', 'sign_stability'], ascending=False
)
direct_teacher.to_csv(OUTPUT_DIR / 'jtbd_direct_teacher_summary.csv', index=False)

print('\nJTBD DECISION CARD')
print(f'Params: factors={JTBD_FACTOR_COUNT_PARAM}; stability={JTBD_STABILITY_RUNS_PARAM}; bootstrap={JTBD_BOOTSTRAP_RUNS_PARAM}')
print(f"Weakest median factor congruence: {factor_stability['median'].min():.3f}")
print('Top direct-teacher job signals: ' + ' | '.join(direct_teacher.head(3).job_family))
print('Recommendation: use factors as validated job families; keep eleven theory-led scores as supporting diagnostics.')


JTBD DECISION TABLE
 factor  median  mean   min
      1   0.984 0.977 0.926
      2   0.995 0.991 0.914
      3   0.994 0.992 0.979
      4   0.991 0.989 0.969
      5   0.978 0.952 0.636
      6   0.987 0.983 0.957
      7   0.877 0.832 0.486

JTBD DECISION CARD
Params: factors=7; stability=20; bootstrap=20
Weakest median factor congruence: 0.877
Top direct-teacher job signals: local_stewardship | relational_support | values_obligation
Recommendation: use factors as validated job families; keep eleven theory-led scores as supporting diagnostics.


## Loop 3 — Shortlist validation and architecture gate


In [19]:
# PASS-2 PARAMS — edit values only, then run this workstream downward.
PASS2_SCENARIOS = ('no_education__dw', 'education_half__dw', 'jtbd_seven_factors')
PASS2_K_RANGE = (3,)                    # Compare the current decision candidate at the same k.
PASS2_RESAMPLES = 'smoke'              # 'smoke', '20', '50', or '100'; use 50 for shortlist validation.
PASS2_MIN_CLUSTER_SHARE = 0.07
FACTOR_MAX_ARI_FOR_MATERIAL_DIFFERENCE = 0.75
FACTOR_MAX_MATCHED_AGREEMENT = 0.80
FACTOR_MAX_TOP_PROFILE_OVERLAP = 0.70


In [20]:
assert PASS2_RESAMPLES in EXPERIMENT_MODES
assert set(PASS2_SCENARIOS).issubset({'no_education__dw', 'education_half__dw', 'writein_structure_low', 'jtbd_seven_factors'})
PASS2_OUTPUT_DIR = OUTPUT_DIR / 'shortlist'
PASS2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Register the fixed seven-factor priority architecture for the factor-input comparison.
factor_count = 7
factor_names = [f'Factor {index}' for index in range(1, factor_count + 1)]
raw_factor_fit = jtbd_fit_latent(jtbd_results['latent_raw'], factor_count)
priority_factor_fit = jtbd_fit_latent(jtbd_results['latent_priority'], factor_count)
priority_loadings, _ = jtbd_match_loadings(raw_factor_fit['loadings'], priority_factor_fit['loadings'])
congruence_matrix = jtbd_loading_congruence(raw_factor_fit['loadings'], priority_factor_fit['loadings'])
_, priority_order = linear_sum_assignment(-congruence_matrix)
prospect_factor_scores = pd.DataFrame(priority_factor_fit['scores'][:, priority_order], index=jtbd_prospects.index, columns=factor_names)
for factor_index, factor_name in enumerate(factor_names):
    sign = np.sign(raw_factor_fit['loadings'][:, factor_index] @ priority_factor_fit['loadings'][:, priority_order[factor_index]]) or 1.0
    prospect_factor_scores[factor_name] *= sign
all_priority_items = jtbd_build_item_matrices(jtbd_retained)[1].loc[jtbd_retained.index, jtbd_results['eligible_features']]
all_factor_scores = project_factor_scores(
    jtbd_results['latent_priority'], all_priority_items,
    priority_factor_fit['corr'], priority_loadings,
)
factor_scores_by_id = all_factor_scores.assign(**{ID_COL: jtbd_retained.loc[all_factor_scores.index, ID_COL].to_numpy()}).set_index(ID_COL)
for factor_name in factor_names:
    unit_name = f'jtbd_factor__{factor_name.replace(" ", "_").lower()}'
    register_external_numeric_unit(
        UNIT_DISTANCE_CACHE, unit_name, 'jtbd_factors',
        retained_model[ID_COL].map(factor_scores_by_id[factor_name]), prospect_fit_mask,
    )

approved_downweights = {
    'giving_attitudes': 0.5,
    'organization_selection_attributes': 0.5,
    'trigger_messages': 0.5,
    'organizations_per_year': 0.5,
}
scenario_configs = {
    'no_education__dw': SegmentExperiment(
        name='no_education__dw', include_blocks=tuple(block for block in BLOCK_NAMES if block != 'education_receptivity'),
        unit_weights=approved_downweights, k_values=PASS2_K_RANGE,
    ),
    'education_half__dw': SegmentExperiment(
        name='education_half__dw', include_blocks=BLOCK_NAMES,
        block_weights={'education_receptivity': 0.5}, unit_weights=approved_downweights,
        k_values=PASS2_K_RANGE,
    ),
    'writein_structure_low': SegmentExperiment(
        name='writein_structure_low', include_blocks=BLOCK_NAMES + (CHARITY_WRITEIN_BLOCK,),
        block_weights={CHARITY_WRITEIN_BLOCK: 0.25}, k_values=PASS2_K_RANGE,
    ),
    'jtbd_seven_factors': SegmentExperiment(
        name='jtbd_seven_factors', include_blocks=('jtbd_factors',), k_values=PASS2_K_RANGE,
    ),
}
pass2_runs = {
    scenario: run_segment_experiment(scenario_configs[scenario], UNIT_DISTANCE_CACHE, PASS2_RESAMPLES)
    for scenario in PASS2_SCENARIOS
}

candidate_rows, profile_cache = [], {}
for scenario, run in pass2_runs.items():
    for k, row in run.candidate_summary.iterrows():
        candidate_rows.append({'scenario': scenario, 'k': int(k), **row.to_dict()})
    tidy, cluster_summary = profile_solution(run, PASS2_K_RANGE[0], PROFILE_VALUES, PROFILE_METADATA)
    profile_cache[scenario] = (tidy, cluster_summary)
    tidy.to_csv(PASS2_OUTPUT_DIR / f'{scenario}_profile_tidy.csv', index=False)
    cluster_summary.to_csv(PASS2_OUTPUT_DIR / f'{scenario}_cluster_summary.csv')

shortlist_decision = pd.DataFrame(candidate_rows).rename(columns={
    'mean_stability_ARI': 'mean_ARI',
    'weakest_cluster_median_jaccard': 'weakest_cluster_jaccard',
    'minimum_cluster_share': 'min_share',
})
shortlist_decision['passes_min_share'] = shortlist_decision.min_share.ge(PASS2_MIN_CLUSTER_SHARE)
shortlist_decision['stability_floor'] = shortlist_decision[['mean_ARI', 'weakest_cluster_jaccard']].min(axis=1)
shortlist_decision.to_csv(OUTPUT_DIR / 'shortlist_decision_table.csv', index=False)

survey_rows = shortlist_decision[shortlist_decision.scenario.ne('jtbd_seven_factors') & shortlist_decision.passes_min_share]
survey_leader_row = survey_rows.sort_values(['stability_floor', 'mean_ARI', 'weakest_cluster_jaccard', 'silhouette'], ascending=False).iloc[0]
survey_leader = survey_leader_row.scenario
factor_row = shortlist_decision[shortlist_decision.scenario.eq('jtbd_seven_factors')].iloc[0]
survey_labels = pass2_runs[survey_leader].candidate_results[int(survey_leader_row.k)].labels
factor_labels = pass2_runs['jtbd_seven_factors'].candidate_results[int(factor_row.k)].labels
matched_pairs, matched_agreement = rectangular_cluster_match(survey_labels, factor_labels)
assignment_ari = adjusted_rand_score(survey_labels, factor_labels)
profile_overlap = top_profile_overlap(profile_cache[survey_leader][0], profile_cache['jtbd_seven_factors'][0], matched_pairs)
factor_more_stable = bool(
    factor_row.mean_ARI > survey_leader_row.mean_ARI
    and factor_row.weakest_cluster_jaccard > survey_leader_row.weakest_cluster_jaccard
)
factor_materially_different = bool(
    assignment_ari < FACTOR_MAX_ARI_FOR_MATERIAL_DIFFERENCE
    and matched_agreement < FACTOR_MAX_MATCHED_AGREEMENT
    and profile_overlap < FACTOR_MAX_TOP_PROFILE_OVERLAP
)
factor_gate = pd.DataFrame([{
    'survey_leader': survey_leader,
    'factor_more_stable_on_both': factor_more_stable,
    'assignment_ARI': assignment_ari,
    'matched_assignment_agreement': matched_agreement,
    'mean_top_profile_overlap': profile_overlap,
    'factor_materially_different': factor_materially_different,
    'factor_eligible': factor_more_stable and factor_materially_different,
}])
factor_gate.to_csv(OUTPUT_DIR / 'factor_architecture_gate.csv', index=False)

print('SHORTLIST DECISION TABLE')
print_decision_table(shortlist_decision, [
    'scenario', 'k', 'weakest_cluster_jaccard', 'mean_ARI', 'min_share',
    'silhouette', 'passes_min_share',
])
print('\nSHORTLIST DECISION CARD')
print(f'Params: scenarios={PASS2_SCENARIOS}; k={PASS2_K_RANGE}; resamples={PASS2_RESAMPLES}')
print(f'Stability-leading survey architecture: {survey_leader}')
print(f'Factor-input more stable on both measures: {factor_more_stable}')
print(f'Factor-input materially different: {factor_materially_different}')
print(f'Factor-input eligible under preregistered gate: {factor_more_stable and factor_materially_different}')
if PASS2_RESAMPLES == 'smoke':
    print('Recommendation: smoke test only — do not select an architecture. Set PASS2_RESAMPLES to 50 for the decision run.')
else:
    print('Recommendation: carry the stability-leading survey architecture into final actionability review before 100-run freeze.')


SHORTLIST DECISION TABLE
          scenario  k  weakest_cluster_jaccard  mean_ARI  min_share  silhouette  passes_min_share
  no_education__dw  3                    0.124     0.289      0.222       0.061              True
education_half__dw  3                    0.480     0.408      0.178       0.057              True
jtbd_seven_factors  3                    0.187     0.184      0.229       0.145              True

SHORTLIST DECISION CARD
Params: scenarios=('no_education__dw', 'education_half__dw', 'jtbd_seven_factors'); k=(3,); resamples=smoke
Stability-leading survey architecture: education_half__dw
Factor-input more stable on both measures: False
Factor-input materially different: True
Factor-input eligible under preregistered gate: False
Recommendation: smoke test only — do not select an architecture. Set PASS2_RESAMPLES to 50 for the decision run.


In [21]:
integrity_gate(
    ENGINE_SNAPSHOT,
    retained_model=retained_model,
    prospect_fit_mask=prospect_fit_mask,
    donor_mask=donor_mask,
)
integrity_gate(SOURCE_SNAPSHOT, raw=raw, retained=retained)

final_outputs = [path for path in OUTPUT_DIR.rglob('*') if path.is_file()]
refactor_manifest = write_manifest(
    'refactored_run',
    {
        'segmentation': {
            'block_weights': SEGMENT_BLOCK_WEIGHTS,
            'unit_downweights': SEGMENT_UNIT_DOWNWEIGHTS,
            'k_range': SEGMENT_K_RANGE,
            'resamples': SEGMENT_RESAMPLES,
        },
        'jtbd': {
            'factor_count': JTBD_FACTOR_COUNT_PARAM,
            'stability_runs': JTBD_STABILITY_RUNS_PARAM,
            'bootstrap_runs': JTBD_BOOTSTRAP_RUNS_PARAM,
        },
        'shortlist': {
            'scenarios': PASS2_SCENARIOS,
            'k_range': PASS2_K_RANGE,
            'resamples': PASS2_RESAMPLES,
        },
    },
    final_outputs,
)
print(f'Integrity gates passed. Wrote {len(final_outputs):,} artifacts plus {refactor_manifest.name}.')


Integrity gates passed. Wrote 91 artifacts plus refactored_run_manifest.json.


# Zone 3 — Archive
Superseded sweeps, optional LLM enrichment, the dormant cause-extension experiment, the old transactional join contract, and the completed spread/redundancy diagnostic were moved to the companion archive notebook. They are intentionally not live here.
